SIMULATION SOLAR BURST SIGNAL BASE ON SFU ON SWS

In [1]:
# %% CELL 0: CẤU HÌNH DỰ ÁN — NGUỒN CHÂN LÝ DUY NHẤT (đường dẫn + tham số chung)
import os
import numpy as np
import sws_to_csv as sws          # bộ chuyển .txt SWS -> .csv + load_day

# ── ĐƯỜNG DẪN (mọi cell lấy từ đây — KHÔNG hardcode rải rác) ──
PROJECT_ROOT = r"E:\1A DATN REAL"
RAW_DIR  = os.path.join(PROJECT_ROOT, "raw_signal_from_sws")            # .txt gốc SWS
CSV_DIR  = os.path.join(PROJECT_ROOT, "file_excel_du_lieu")             # .csv (mở Excel)
IQ_DIR   = os.path.join(PROJECT_ROOT, "IQ_data_for signal_processing")  # hex 16-bit (Cell 4)
BEAM_DIR = os.path.join(PROJECT_ROOT, "file_trichxuat_primary beam_cua_anten") # .npz beam NEC
IMG_DIR  = os.path.join(PROJECT_ROOT, "Cell_I_Data")                  # ảnh + .npz Cell H/I
for _d in (CSV_DIR, IQ_DIR, IMG_DIR):
    os.makedirs(_d, exist_ok=True)

sws.BASE_DIR = RAW_DIR            # load_day đọc .txt ở đây, xuất .csv sang CSV_DIR
NPZ_BEAM = os.path.join(BEAM_DIR, "UFRAME_primary_beam_610MHz.npz")

# ── CHỌN DỮ LIỆU: CHỈ CẦN ĐỔI 2 DÒNG NÀY ──
NGAY = "20-3-2025"               # tên file .txt trong raw_signal_from_sws (không gồm .txt)
FREQ = "F610"                    # F245 F410 F610 F1415 F2695 F4995 F8800 F15400

# ── THAM SỐ DÙNG CHUNG (đổi ở đây, mọi cell theo) ──
N_SAMPLES        = 50_000        # số mẫu chuỗi điện áp (= số hàng V_matrix)
fc               = 610e6         # Hz (phải khớp FREQ ở trên)
BW_Hz            = 1e6
R_load           = 75.0          # Ω — trở kháng tải đầu ra anten UFRAME
T_sys            = 150.0
INTEGRATION_BINS = 50
LAT_DEG          = -22.22         # Learmonth
DEC_SUN_DEG      =  0.0          # xuân phân 20-3-2025: δ ≈ 0°

print(f"Cell 0: cấu hình OK | ngày {NGAY}, {FREQ} | gốc: {PROJECT_ROOT}")

Cell 0: cấu hình OK | ngày 20-3-2025, F610 | gốc: E:\1A DATN REAL


In [2]:
import csv
import os
import numpy as np
import matplotlib.pyplot as plt

print("Cell 1: Đã nạp thư viện thành công!")

Cell 1: Đã nạp thư viện thành công!


In [3]:
# %% CELL 2: NẠP DỮ LIỆU BAN NGÀY TỪ SWS (đổi ngày/tần số ở Cell 0)
# load_day: đọc {NGAY}.txt trong RAW_DIR, tự bỏ phần ban đêm (//////), trả về flux
# ban ngày, đồng thời xuất {NGAY}.csv sang CSV_DIR để xem bằng Excel.
t_raw, sfu_raw = sws.load_day(NGAY, freq=FREQ, out_dir=CSV_DIR)
print(f"Cell 2 Xong! {len(sfu_raw)} mẫu ban ngày ({FREQ}, ngày {NGAY}).")

[20-3-2025] F610: khoi ban ngay 0-37320 (~10.4 h, 37320 mau) | flux tb 68.3
Cell 2 Xong! 37320 mẫu ban ngày (F610, ngày 20-3-2025).


In [4]:
# %% CELL 2.5: NẠP PRIMARY BEAM THẬT TỪ 4NEC2  (tích hợp giản đồ bức xạ NEC)
# Thay mô hình Gaussian lý tưởng bằng giản đồ bức xạ THẬT của anten UFRAME,
# trích từ mô phỏng 4nec2 -> file "E:\1A DATN REAL\file_trichxuat_primary beam_cua_anten\UFRAME_primary_beam_610MHz.npz" (xuất bởi parse_nec_out.py).
import numpy as np
from scipy.interpolate import RegularGridInterpolator

class PrimaryBeam:
    """Nạp giản đồ bức xạ NEC (.npz) và tra cứu đáp ứng theo góc lệch boresight.
    Tự tìm boresight = điểm gain cực đại nên chạy đúng với mọi quy ước góc."""
    def __init__(self, npz_path):
        d = np.load(npz_path)
        self.theta    = np.asarray(d["theta_deg"], float)
        self.phi      = np.asarray(d["phi_deg"], float)
        self.gain_dBi = np.asarray(d["gain_dBi"], float)
        self.freq_MHz = float(d["freq_MHz"]) if "freq_MHz" in d else None
        i, j = np.unravel_index(np.nanargmax(self.gain_dBi), self.gain_dBi.shape)
        self.boresight    = (self.theta[i], self.phi[j])
        self.gain_max_dBi = float(self.gain_dBi[i, j])
        # Đáp ứng công suất chuẩn hóa boresight = 1.0
        self.A_power = 10.0 ** ((self.gain_dBi - self.gain_max_dBi) / 10.0)
        self._interp = RegularGridInterpolator(
            (self.theta, self.phi), self.A_power, bounds_error=False, fill_value=0.0)

    def response_power(self, theta_deg, phi_deg):
        theta_deg = np.atleast_1d(np.asarray(theta_deg, float))
        phi_deg   = np.atleast_1d(np.asarray(phi_deg, float))
        return self._interp(np.stack([theta_deg, phi_deg], axis=-1))

    def response_voltage_offset(self, delta_deg):
        """Đáp ứng BIÊN ĐỘ ĐIỆN ÁP theo góc lệch khỏi tâm búp (boresight)."""
        th0, ph0 = self.boresight
        delta_deg = np.atleast_1d(np.asarray(delta_deg, float))
        return np.sqrt(self.response_power(th0 + delta_deg, np.full_like(delta_deg, ph0)))

NPZ_PATH = NPZ_BEAM   # từ Cell 0
BEAM = PrimaryBeam(NPZ_PATH)
print(f"Cell 2.5: Đã nạp primary beam THẬT — {BEAM.freq_MHz:.0f} MHz, "
      f"gain đỉnh {BEAM.gain_max_dBi:.2f} dBi (thay cho mô hình Gaussian)")
print(f"  File: {NPZ_PATH}")

Cell 2.5: Đã nạp primary beam THẬT — 610 MHz, gain đỉnh 9.48 dBi (thay cho mô hình Gaussian)
  File: E:\1A DATN REAL\file_trichxuat_primary beam_cua_anten\UFRAME_primary_beam_610MHz.npz


In [5]:
# %% CELL A (= Cell 3): VẬT LÝ HỆ THỐNG + OBSERVATION + BIÊN ĐỘ TÍN HIỆU
import interfsim as ix
from scipy.interpolate import CubicSpline
from math import degrees

# ── 1. HẰNG SỐ + DẪN XUẤT (N_SAMPLES, fc, BW_Hz, R_load, T_sys, INTEGRATION_BINS
#     lấy từ Cell 0 — không khai báo lại ở đây) ──
c     = 299_792_458.0
k_B   = 1.38064852e-23
lam   = c / fc
G_lin = 10 ** (BEAM.gain_max_dBi / 10.0)          # gain THẬT từ NEC
Aeff  = G_lin * lam**2 / (4 * np.pi)              # Ae = G·λ²/4π  [Friis]

# ── 2. NỘI SUY FLUX BAN NGÀY LÊN LƯỚI MÔ PHỎNG (CubicSpline) ──
sfu_daylight = sfu_raw                            # ĐÃ là ban ngày (load_day bỏ đêm) — không cắt [0:43200]
t_src    = np.linspace(0, 12, len(sfu_daylight))  # co ~10h thật vào cửa sổ transit
t_grid   = np.linspace(0, 12, N_SAMPLES)
sfu_data = np.clip(CubicSpline(t_src, sfu_daylight)(t_grid), 0.0, None)

# ── 3. OBSERVATION ──
dec_sun = np.radians(DEC_SUN_DEG)
H_deg   = np.linspace(-90.0, 90.0, N_SAMPLES)
obs = ix.Observation(lat_deg=LAT_DEG, dec_deg=float(degrees(dec_sun)),
                     fc_hz=fc, H_deg=H_deg)

# ── 4. BIÊN ĐỘ VẬT LÝ (Friis + primary beam NEC) ──
sfu_si   = sfu_data * 1e-22                        # 1 SFU = 1e-22 W/m²/Hz
P_signal = sfu_si * Aeff * BW_Hz
V_rms    = np.sqrt(P_signal * R_load)
beam_zenith = BEAM.response_voltage_offset(ix.zenith_angle_deg(obs))
amp_env  = V_rms * beam_zenith

# ── 5. NHIỄU JOHNSON–NYQUIST ──
P_noise     = k_B * T_sys * BW_Hz
V_noise     = np.sqrt(P_noise * R_load)
noise_sigma = V_noise / np.sqrt(2)                # mỗi thành phần I/Q

# ── 6. QUÁ TRÌNH NGUỒN (coherent far-field, seed cố định) ──
rng = np.random.default_rng(42)
source_process = (rng.standard_normal(N_SAMPLES) + 1j*rng.standard_normal(N_SAMPLES)) / np.sqrt(2)

# ── [Validate] ──
SNR_peak_dB = 10*np.log10(np.max(P_signal) * R_load / P_noise)
print(f"[Validate] Peak SNR transit: {SNR_peak_dB:.1f} dB | "
      f"V_sig_peak={np.max(V_rms)*1e6:.2f} µV, V_noise={V_noise*1e6:.2f} µV")
print(f"  Aeff={Aeff*1e4:.2f} cm² | T_sys={T_sys} K | BW={BW_Hz/1e6:.0f} MHz | N={N_SAMPLES}")

[Validate] Peak SNR transit: 18.4 dB | V_sig_peak=0.38 µV, V_noise=0.39 µV
  Aeff=1705.19 cm² | T_sys=150.0 K | BW=1 MHz | N=50000


In [6]:
# %% CELL B: CẤU HÌNH MẢNG ANTEN — nơi DUY NHẤT chỉnh vị trí anten
import interfsim as ix

# ── Chọn một layout (bỏ comment dòng muốn dùng) ───────────────────────────
ARRAY = ix.preset_compact()              # 4 anten Đông-Tây [0,8,16,24] m  ← mặc định
# ARRAY = ix.preset_extended()           # chữ thập 2D graded, baseline 5–130 m
# ARRAY = ix.linear(8, 15, az_deg=0)    # 8 anten hướng Bắc, bước 15 m
# ARRAY = ix.tee(6, 5, 18)              # chữ T: 6 Đông-Tây + 5 Bắc, bước 18 m
# ARRAY = ix.random_2d(16, 120, seed=3) # 16 anten ngẫu nhiên trong 120×120 m
# ARRAY = ix.from_enu([[0,0],[10,0],[10,10],[0,10]])  # tuỳ ý (E,N) tính bằng mét

print(ARRAY.summary())
print("Pairs          :", ARRAY.pairs)
print("Baseline lengths (m):", np.round(ARRAY.baseline_lengths, 1))

# Alias tương thích cho Cell 4 (hex export)
NUM_ANTENNAS = ARRAY.n_ant
d_spacing    = float(ARRAY.baseline_lengths.min())


compact_outdoor_EW: N=4 anten, 6 baseline, baseline min/max = 8.0/24.0 m
Pairs          : [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
Baseline lengths (m): [ 8. 16. 24.  8. 16.  8.]


In [7]:
# %% CELL C: SINH CHUỖI ĐIỆN ÁP PER-ANTEN (Engine A — simulate_voltages)
import interfsim as ix

V_matrix = ix.simulate_voltages(
    ARRAY, obs,
    amplitude_envelope = amp_env,
    noise_sigma        = noise_sigma,
    source_process     = source_process,
)

print(f"V_matrix shape : {V_matrix.shape}  (samples × antennas)  dtype: {V_matrix.dtype}")
print(f"  |V| max = {np.max(np.abs(V_matrix))*1e6:.4f} µV  (dùng cho scale FPGA ở Cell 4)")


V_matrix shape : (50000, 4)  (samples × antennas)  dtype: complex128
  |V| max = 1.5048 µV  (dùng cho scale FPGA ở Cell 4)


In [8]:
# %% CELL 4: LƯỢNG TỬ HÓA 16-BIT + XUẤT HEX CHO MẢNG N ANTEN
import os, datetime

output_dir    = IQ_DIR                                   # từ Cell 0
os.makedirs(output_dir, exist_ok=True)
filename      = os.path.join(output_dir, "iq_array_stimulus.txt")
filename_meta = os.path.join(output_dir, "iq_array_stimulus_meta.txt")

# Lấy kích thước TRỰC TIẾP từ V_matrix (tự đồng bộ, không hardcode N_SAMPLES/NUM_ANTENNAS)
V = np.atleast_2d(V_matrix)
N_rows, N_ant = V.shape
print(f"[Sync] V_matrix = {N_rows} mẫu × {N_ant} anten")

max_val = np.nanmax(np.abs(V))
scale_factor = 1.0 if (not np.isfinite(max_val) or max_val == 0) else 32767.0/max_val
print(f"[Validate] max|V|={max_val*1e6:.4f} µV | scale={scale_factor:.4e} (1 LSB={max_val/32767*1e6:.6f} µV)")
lsb = V_noise * scale_factor
print(f"[Validate] V_noise={V_noise*1e6:.4f} µV -> {lsb:.2f} LSB "
      f"({'OK >= 3 LSB' if lsb >= 3 else 'WARN: nhiễu dưới 3 LSB'})")

re16 = np.round(V.real * scale_factor).astype(np.int16)
im16 = np.round(V.imag * scale_factor).astype(np.int16)

with open(filename, "w", encoding="utf-8") as f:
    for n in range(N_rows):
        f.write(" ".join(f"{int(re16[n,k])&0xFFFF:04x} {int(im16[n,k])&0xFFFF:04x}"
                         for k in range(N_ant)) + "\n")

with open(filename_meta, "w", encoding="utf-8") as f:
    f.write("# IQ Array Stimulus - Metadata\n")
    f.write(f"# Generated : {datetime.datetime.utcnow().isoformat()} UTC\n")
    f.write(f"# Data day  : {NGAY}   Freq : {FREQ}   Source : SWS Learmonth RSTN\n#\n")
    f.write(f"n_samples     = {N_rows}\n")
    f.write(f"num_antennas  = {N_ant}\n")
    f.write(f"array         = {ARRAY.name}\n")
    f.write(f"fc_hz         = {fc:.6e}\n")
    f.write(f"bw_hz         = {BW_Hz:.6e}\n")
    f.write(f"t_sys_k       = {T_sys:.2f}\n")
    f.write(f"r_load_ohm    = {R_load:.1f}\n")
    f.write(f"aeff_m2       = {Aeff:.6e}\n")
    f.write(f"scale_factor  = {scale_factor:.10e}\n")
    f.write(f"max_val_volt  = {max_val:.10e}\n")
    f.write(f"v_noise_volt  = {V_noise:.10e}\n")
    f.write(f"snr_peak_db   = {SNR_peak_dB:.4f}\n")
    f.write(f"dec_sun_rad   = {dec_sun:.6f}\n")
    f.write("bits          = 16\n")
    f.write(f"format        = (I Q) x {N_ant} anten, hex 16-bit 2's complement\n")

print(f"\nCell 4 Xong! Hex: {filename}\n  {N_rows} hàng x {N_ant*2} cột (I/Q mỗi anten)")

[Sync] V_matrix = 50000 mẫu × 4 anten
[Validate] max|V|=1.5048 µV | scale=2.1775e+10 (1 LSB=0.000046 µV)
[Validate] V_noise=0.3941 µV -> 8581.76 LSB (OK >= 3 LSB)

Cell 4 Xong! Hex: E:\1A DATN REAL\IQ_data_for signal_processing\iq_array_stimulus.txt
  50000 hàng x 8 cột (I/Q mỗi anten)


C:\Users\admin\AppData\Local\Temp\ipykernel_4776\2349113956.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f.write(f"# Generated : {datetime.datetime.utcnow().isoformat()} UTC\n")


In [9]:
# %% CELL 5: KIỂM TRA TRỰC QUAN - TRỤC THỜI GIAN THỰC (PLOTLY)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime, timedelta

# ──────────────────────────────────────────────────────────────────────────────
# 1. TRỤC THỜI GIAN (06:00 → 18:00)
# ──────────────────────────────────────────────────────────────────────────────
# 2024-04-28 at Learmonth (UTC+8 local), Sun rises ~22:20 UTC previous day,
# transits ~03:55 UTC, sets ~09:30 UTC. Mining 12h window for visualization simplicity.
start_time = datetime(2024, 4, 28, 0, 0, 0)
time_labels = [(start_time + timedelta(seconds=int(s))).strftime('%H:%M:%S')
               for s in np.linspace(0, 12*3600, N_SAMPLES)]

# ──────────────────────────────────────────────────────────────────────────────
# 2. FIX CHÍNH: TÍNH ENVELOPE bằng sliding-window RMS thay vì raw |V|
#    Raw |V| ở 50000 điểm → Plotly downsample ngẫu nhiên → mất hình dạng transit
#    RMS window làm nổi bật beam envelope + burst, suppress noise spike lẻ
# ──────────────────────────────────────────────────────────────────────────────
WINDOW = 500   # ~500 mẫu = 0.12h = ~7 phút sliding window

def sliding_rms(x, w):
    """RMS trượt bằng cumsum — O(N), nhanh hơn np.convolve cho array lớn."""
    x2 = np.abs(x) ** 2
    cs = np.cumsum(x2)
    cs = np.concatenate([[0], cs])
    rms = np.sqrt((cs[w:] - cs[:-w]) / w)
    # Pad đầu để giữ đúng N_SAMPLES
    pad = np.full(w - 1, rms[0])
    return np.concatenate([pad, rms])

# Tính RMS envelope cho antenna 1 (index 0)
V_ant1_rms = sliding_rms(V_matrix[:, 0], WINDOW)

# Đơn vị: chuyển sang µV để dễ đọc trên trục Y
V_ant1_uV  = V_ant1_rms * 1e6          # µV
V_noise_uV = V_noise * 1e6             # µV — đường noise floor tham chiếu

# ──────────────────────────────────────────────────────────────────────────────
# 3. TẠO SUBPLOTS
# ──────────────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.14,
    subplot_titles=(
        "Nang luong Mat Troi Goc (Du lieu 12h ban ngay)",
        f"Envelope RMS tin hieu mo phong (Ang-ten 1) — cua so {WINDOW} mau"
    )
)

# ──────────────────────────────────────────────────────────────────────────────
# 4. GRAPH 1 — SFU raw
# ──────────────────────────────────────────────────────────────────────────────
fig.add_trace(
    go.Scatter(
        x=time_labels, y=sfu_data,
        mode='lines', name='SFU',
        line=dict(color='darkorange', width=1.2),
        hovertemplate='%{x}<br>Flux: %{y:.2f} SFU<extra></extra>'
    ),
    row=1, col=1
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. GRAPH 2 — RMS envelope + noise floor reference
# ──────────────────────────────────────────────────────────────────────────────
# 5a. Vùng tô màu dưới envelope (area fill)
fig.add_trace(
    go.Scatter(
        x=time_labels, y=V_ant1_uV,
        mode='lines', name='RMS envelope |V1|',
        fill='tozeroy',
        fillcolor='rgba(65, 105, 225, 0.15)',
        line=dict(color='royalblue', width=1.5),
        hovertemplate='%{x}<br>RMS: %{y:.4f} uV<extra></extra>'
    ),
    row=2, col=1
)

# 5b. Đường noise floor vật lý (từ Cell 3: k_B * T_sys * BW * R_load)
fig.add_hline(
    y=V_noise_uV,
    line=dict(color='red', width=1.2, dash='dash'),
    annotation_text=f"Noise floor: {V_noise_uV:.4f} uV (T_sys={T_sys}K)",
    annotation_position="top right",
    annotation_font_size=11,
    row=2, col=1
)

# 5c. Đường peak signal tại transit (θ=0)
transit_idx   = N_SAMPLES // 2
V_transit_uV  = V_ant1_uV[transit_idx]
fig.add_hline(
    y=V_transit_uV,
    line=dict(color='green', width=1.0, dash='dot'),
    annotation_text=f"Transit peak ~{V_transit_uV:.4f} uV",
    annotation_position="bottom right",
    annotation_font_size=11,
    row=2, col=1
)

# ──────────────────────────────────────────────────────────────────────────────
# 6. LAYOUT
# ──────────────────────────────────────────────────────────────────────────────
fig.update_xaxes(nticks=13, gridcolor='lightgray', row=1, col=1)
fig.update_xaxes(
    title_text="Thoi gian quan sat (06:00 den 18:00)",
    nticks=13, gridcolor='lightgray',
    row=2, col=1
)
fig.update_yaxes(title_text="Mat do Flux (SFU)", gridcolor='lightgray', row=1, col=1)
fig.update_yaxes(title_text="RMS Envelope (uV)",  gridcolor='lightgray', row=2, col=1)

fig.update_layout(
    height=850,
    title_text="PHAN TICH CHU KY QUA CANH MAT TROI 610MHz (06:00 - 18:00)",
    title_x=0.5,
    hovermode="x unified",
    template="plotly_white",
    showlegend=False
)

fig.show()

# ──────────────────────────────────────────────────────────────────────────────
# 7. VALIDATE IN RA
# ──────────────────────────────────────────────────────────────────────────────
snr_visual = V_transit_uV / V_noise_uV
print(f"Cell 5 Xong!")
print(f"  RMS window     : {WINDOW} mau (~{WINDOW/N_SAMPLES*12*60:.1f} phut)")
print(f"  Noise floor    : {V_noise_uV:.4f} uV")
print(f"  Transit peak   : {V_transit_uV:.4f} uV")
print(f"  SNR visual     : {snr_visual:.1f}x  ({20*np.log10(snr_visual):.1f} dB)")

Cell 5 Xong!
  RMS window     : 500 mau (~7.2 phut)
  Noise floor    : 0.3941 uV
  Transit peak   : 0.4439 uV
  SNR visual     : 1.1x  (1.0 dB)


In [10]:
# %% CELL E: CORRELATOR — TƯƠNG QUAN CHÉO + TÍCH LUỸ
import interfsim as ix
import plotly.graph_objects as go
from plotly.subplots import make_subplots

visibilities, centers = ix.correlate(V_matrix, ARRAY, INTEGRATION_BINS)
# visibilities: (NI, NB)  centers: chỉ số thời gian giữa mỗi dump

NI = visibilities.shape[0]
NB = visibilities.shape[1]
vis_amp   = np.abs(visibilities)
vis_phase = np.angle(visibilities, deg=True)

# Smooth biên độ bằng sliding mean (giữ hình dạng bell curve transit)
def sliding_mean(x, w):
    cs  = np.cumsum(x); cs = np.concatenate([[0], cs])
    out = (cs[w:] - cs[:-w]) / w
    return np.concatenate([np.full(w - 1, out[0]), out])

SMOOTH_W = 10
vis_amp_smooth = np.stack(
    [sliding_mean(vis_amp[:, b], SMOOTH_W) for b in range(NB)], axis=1)

# Nhãn baseline từ ARRAY (tự cập nhật khi đổi layout ở Cell B)
bl_labels = [f"Ant {i}-{j} ({ARRAY.baseline_lengths[b]:.0f}m)"
             for b, (i, j) in enumerate(ARRAY.pairs)]

time_axis_vis = np.linspace(6, 18, NI)
colors = ['royalblue','darkorange','green','red','purple','brown',
          'teal','magenta','olive','navy']

# ── Baseline Ant 0-1: luôn là pair đầu tiên trong danh sách ──
IDX_01   = ARRAY.pairs.index((0, 1))
label_01 = bl_labels[IDX_01]

figE = make_subplots(
    rows=4, cols=1,
    vertical_spacing=0.09,
    subplot_titles=(
        "Biên độ Visibility |V_ij| — tất cả baseline (smooth)",
        "Pha Visibility (°) — tất cả baseline",
        f"Biên độ Visibility |V_01| — chỉ {label_01} (raw + smooth)",
        f"Pha Visibility (°) — chỉ {label_01}",
    )
)

# ── Graph 1 & 2: tất cả baseline ──
for b in range(NB):
    col = colors[b % len(colors)]
    figE.add_trace(go.Scatter(
        x=time_axis_vis, y=vis_amp_smooth[:, b],
        mode='lines', name=bl_labels[b], line=dict(color=col, width=1.5),
        hovertemplate='%{x:.2f}h<br>|V|=%{y:.4e}<extra>' + bl_labels[b] + '</extra>'
    ), row=1, col=1)
    figE.add_trace(go.Scatter(
        x=time_axis_vis, y=vis_phase[:, b],
        mode='lines', name=bl_labels[b], showlegend=False,
        line=dict(color=col, width=0.8),
        hovertemplate='%{x:.2f}h<br>pha=%{y:.1f}°<extra>' + bl_labels[b] + '</extra>'
    ), row=2, col=1)

# ── Graph 3: biên độ Ant 0-1 riêng (raw mờ + smooth đậm + fill) ──
figE.add_trace(go.Scatter(
    x=time_axis_vis, y=vis_amp[:, IDX_01],
    mode='lines', name="|V_01| raw",
    line=dict(color='lightsteelblue', width=0.7),
    hovertemplate='%{x:.2f}h<br>|V| raw=%{y:.4e}<extra></extra>'
), row=3, col=1)
figE.add_trace(go.Scatter(
    x=time_axis_vis, y=vis_amp_smooth[:, IDX_01],
    mode='lines', name="|V_01| smooth",
    line=dict(color='royalblue', width=2.0),
    fill='tozeroy', fillcolor='rgba(65,105,225,0.12)',
    hovertemplate='%{x:.2f}h<br>|V| smooth=%{y:.4e}<extra></extra>'
), row=3, col=1)

# ── Graph 4: pha Ant 0-1 riêng ──
figE.add_trace(go.Scatter(
    x=time_axis_vis, y=vis_phase[:, IDX_01],
    mode='lines', name="Pha V_01",
    line=dict(color='darkorange', width=1.0),
    hovertemplate='%{x:.2f}h<br>pha=%{y:.1f}°<extra></extra>'
), row=4, col=1)

# ── Layout chung ──
figE.update_xaxes(gridcolor='lightgray')
figE.update_xaxes(title_text="Thời gian (giờ)", gridcolor='lightgray', row=4, col=1)
figE.update_yaxes(title_text="|V_ij| (V²)",  gridcolor='lightgray', row=1, col=1)
figE.update_yaxes(title_text="Pha (°)", gridcolor='lightgray',
                  tickvals=[-180,-90,0,90,180], row=2, col=1)
figE.update_yaxes(title_text="|V_01| (V²)", gridcolor='lightgray', row=3, col=1)
figE.update_yaxes(title_text="Pha_01 (°)", gridcolor='lightgray',
                  tickvals=[-180,-90,0,90,180], row=4, col=1)
figE.update_layout(
    height=1300,
    title_text=f"VISIBILITIES — {ARRAY.name}  (correlator output, dump={INTEGRATION_BINS} mẫu)",
    title_x=0.5, hovermode="x unified", template="plotly_white",
    legend=dict(orientation="h", y=1.03)
)
figE.show()

dt_sample = 12 * 3600 / N_SAMPLES
dump_time = INTEGRATION_BINS * dt_sample
print(f"\nCell E Xong!  Vis {visibilities.shape}  dump_time={dump_time:.1f} s/integration")

# [Validate] Van Cittert-Zernike: baseline dài hơn → |V| nhỏ hơn tại đỉnh transit
transit_bin   = NI // 2
peak_per_bl   = vis_amp_smooth[transit_bin, :]
print("[Validate] Biên độ Visibility tại đỉnh transit (Van Cittert-Zernike):")
for b in range(NB):
    flag = ('OK' if b == 0 or peak_per_bl[b] <= peak_per_bl[b-1]
            else 'WARN: nên giảm theo baseline')
    print(f"  {bl_labels[b]:22s} | {peak_per_bl[b]:.4e}  {flag}")
print(f"\n[Ant 0-1 detail] peak |V_01| = {vis_amp_smooth[transit_bin, IDX_01]:.4e}"
      f"  | pha tại transit = {vis_phase[transit_bin, IDX_01]:.1f}°")



Cell E Xong!  Vis (1000, 6)  dump_time=43.2 s/integration
[Validate] Biên độ Visibility tại đỉnh transit (Van Cittert-Zernike):
  Ant 0-1 (8m)           | 5.3364e-14  OK
  Ant 0-2 (16m)          | 4.4627e-14  OK
  Ant 0-3 (24m)          | 5.4303e-14  WARN: nên giảm theo baseline
  Ant 1-2 (8m)           | 4.7505e-14  OK
  Ant 1-3 (16m)          | 5.2023e-14  WARN: nên giảm theo baseline
  Ant 2-3 (8m)           | 4.8958e-14  OK

[Ant 0-1 detail] peak |V_01| = 5.3364e-14  | pha tại transit = -26.0°


In [11]:
# %% CELL F: BỐ TRÍ ĐỊA LÝ + U-V COVERAGE (ENU → XYZ → u,v,w — TMS Eq. 4.1)
import interfsim as ix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ── 1. TÍNH U,V,W — giữ nguyên để Cell G dùng ───────────────────────────────
obs_centers = ix.Observation(
    lat_deg = obs.lat_deg,
    dec_deg = obs.dec_deg,
    fc_hz   = obs.fc_hz,
    H_deg   = obs.H_deg[centers],
)
u, v, w = ix.compute_uvw(ARRAY.baseline_vectors_enu, obs_centers)

# [Validate] — so sánh dump gần transit với compute_uvw(H=0)
transit_vis = NI // 2
obs_h0 = ix.Observation(obs.lat_deg, obs.dec_deg, obs.fc_hz, H_deg=np.array([0.0]))
u0, v0, _ = ix.compute_uvw(ARRAY.baseline_vectors_enu, obs_h0)
H_transit  = obs.H_deg[centers[transit_vis]]
print(f"[Validate] Dump #{transit_vis} (H={H_transit:.3f}°) vs tham chiếu compute_uvw(H=0):")
for b in range(NB):
    du   = abs(u[transit_vis, b] - float(u0[0, b]))
    dv   = abs(v[transit_vis, b] - float(v0[0, b]))
    flag = 'OK' if du < 1.0 and dv < 0.5 else 'WARN'
    print(f"  {bl_labels[b]:22s} | u={u[transit_vis,b]:8.3f} (ref={float(u0[0,b]):8.3f})"
          f"  v={v[transit_vis,b]:6.3f} (ref={float(v0[0,b]):6.3f})  {flag}")

# ════════════════════════════════════════════════════════════════════════════
# 2. HÌNH 1 — QUẢ ĐỊA CẦU TƯƠNG TÁC (orthographic projection)
# ════════════════════════════════════════════════════════════════════════════
LAT_OBS  = obs.lat_deg
LON_OBS  = 114.1
DEC_SUN  = obs.dec_deg
lam_m    = obs.lam

H_arr    = obs.H_deg[centers]
lon_sun  = ((LON_OBS - H_arr + 180) % 360) - 180
lon_full = np.linspace(-180, 180, 361)

# Tính thời lượng quan sát để hiển thị trong legend
H_span_h = (H_arr.max() - H_arr.min()) / 15.0   # giờ

fig_globe = go.Figure()

fig_globe.add_trace(go.Scattergeo(
    lat=np.zeros(361), lon=lon_full, mode='lines',
    line=dict(color='rgba(220,220,100,0.5)', width=1),
    name='Xích đạo'
))
fig_globe.add_trace(go.Scattergeo(
    lat=np.full(361, DEC_SUN), lon=lon_full, mode='lines',
    line=dict(color='gold', width=1.5, dash='dot'),
    name=f'Xích vĩ Mặt Trời cả ngày (δ = {DEC_SUN:.1f}°)'
))
fig_globe.add_trace(go.Scattergeo(
    lat=np.full(len(H_arr), DEC_SUN), lon=lon_sun, mode='lines',
    line=dict(color='darkorange', width=5),
    name=(f'Cửa sổ quan sát: H = {H_arr.min():.0f}° → {H_arr.max():.0f}°  '
          f'({H_span_h:.1f} giờ, từ mọc đến lặn)')
))
fig_globe.add_trace(go.Scattergeo(
    lat=[DEC_SUN], lon=[LON_OBS], mode='markers+text',
    marker=dict(size=14, color='yellow', symbol='star'),
    text=['  Transit (H=0)'], textfont=dict(color='yellow', size=10),
    textposition='middle right',
    name='Transit (H=0)'
))
fig_globe.add_trace(go.Scattergeo(
    lat=[LAT_OBS, DEC_SUN], lon=[LON_OBS, LON_OBS], mode='lines',
    line=dict(color='rgba(0,230,230,0.8)', width=2, dash='dash'),
    name='Hướng nhìn transit'
))
fig_globe.add_trace(go.Scattergeo(
    lat=[LAT_OBS], lon=[LON_OBS], mode='markers+text',
    marker=dict(size=20, color='red', symbol='star'),
    text=[f'  Learmonth<br>  ({abs(LAT_OBS):.2f}°S, {LON_OBS:.1f}°E)'],
    textfont=dict(color='white', size=11),
    textposition='middle right',
    name='Đài quan sát Learmonth'
))
fig_globe.add_trace(go.Scattergeo(
    lat=[21.03], lon=[105.85], mode='markers+text',
    marker=dict(size=16, color='#00ffcc', symbol='diamond'),
    text=['  Hà Nội, Việt Nam<br>  (21.0°N, 105.9°E)'],
    textfont=dict(color='#00ffcc', size=11),
    textposition='middle right',
    name='Việt Nam (Hà Nội)'
))

fig_globe.update_geos(
    projection_type='orthographic',
    center=dict(lat=0, lon=108),
    showland=True,       landcolor='rgb(55,85,55)',
    showocean=True,      oceancolor='rgb(15,45,100)',
    showcoastlines=True, coastlinecolor='rgba(200,200,200,0.7)', coastlinewidth=0.8,
    showlakes=False,
    lataxis_showgrid=True, lonaxis_showgrid=True,
    lataxis_gridcolor='rgba(255,255,255,0.08)',
    lonaxis_gridcolor='rgba(255,255,255,0.08)',
    bgcolor='rgb(8,8,25)',
)
fig_globe.update_layout(
    title=dict(
        text=(
            f'QUẢ ĐỊA CẦU — Đài Learmonth (lat={LAT_OBS}°, lon={LON_OBS}°E)  ·  '
            f'Quỹ đạo biểu kiến Mặt Trời (δ={DEC_SUN:.1f}°)<br>'
            f'<sup>'
            f'Cửa sổ cam = toàn bộ {H_span_h:.1f} giờ từ mọc đến lặn '
            f'(H = ±{abs(H_arr).max():.0f}° = ±{abs(H_arr).max()/15:.1f}h).  '
            f'Trái Đất tự quay 15°/h → baseline chiếu lên bầu trời theo ellipse → tạo u-v coverage'
            f'</sup>'
        ),
        x=0.5, font=dict(size=13)
    ),
    height=640, template='plotly_dark',
    legend=dict(x=0.01, y=0.02, font=dict(color='white', size=10),
                bgcolor='rgba(0,0,0,0.6)', bordercolor='gray', borderwidth=1)
)
fig_globe.show()

# ════════════════════════════════════════════════════════════════════════════
# 3. HÌNH 2 — BỐ TRÍ ANTEN (ENU, mét) + U-V COVERAGE (bước sóng)
# ════════════════════════════════════════════════════════════════════════════
colors = ['royalblue','darkorange','green','red','purple','brown',
          'teal','magenta','olive','navy']

figF = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.13,
    subplot_titles=(
        (f'Bố trí anten tại Learmonth — tọa độ ENU (mét)<br>'
         f'<sup>Hướng Đông-Tây · spacing={ARRAY.baseline_lengths.min():.0f} m · '
         f'baseline max={ARRAY.baseline_lengths.max():.0f} m</sup>'),
        (f'U-V Coverage (đơn vị bước sóng λ = {lam_m*100:.1f} cm tại {obs.fc_hz/1e6:.0f} MHz)<br>'
         f'<sup>1 λ = {lam_m*100:.1f} cm  →  baseline max = {ARRAY.baseline_lengths.max():.0f} m '
         f'= {ARRAY.baseline_lengths.max()/lam_m:.1f} λ  ·  ENU→XYZ→(u,v,w) theo TMS Eq.4.1</sup>'),
    )
)

pos_E = ARRAY.positions_enu[:, 0]
pos_N = ARRAY.positions_enu[:, 1]
pad   = max(3.0, ARRAY.baseline_lengths.min() * 0.3)

for b, (i, j) in enumerate(ARRAY.pairs):
    figF.add_trace(go.Scatter(
        x=[pos_E[i], pos_E[j]], y=[pos_N[i], pos_N[j]],
        mode='lines', line=dict(color=colors[b % len(colors)], width=2.5),
        name=f'{bl_labels[b]}', legendgroup=f'bl{b}',
        hovertemplate=f'{bl_labels[b]} = {ARRAY.baseline_lengths[b]:.0f} m<extra></extra>'
    ), row=1, col=1)

for k in range(ARRAY.n_ant):
    figF.add_trace(go.Scatter(
        x=[pos_E[k]], y=[pos_N[k]], mode='markers+text',
        marker=dict(size=22, color='white', symbol='square',
                    line=dict(color='navy', width=2.5)),
        text=[f'Ant {k}'], textposition='top center',
        textfont=dict(size=10, color='navy'),
        showlegend=False,
        hovertemplate=f'Ant {k}: E={pos_E[k]:.1f} m, N={pos_N[k]:.1f} m<extra></extra>'
    ), row=1, col=1)

for k in range(ARRAY.n_ant - 1):
    mid_E = (pos_E[k] + pos_E[k+1]) / 2
    dist  = abs(pos_E[k+1] - pos_E[k])
    figF.add_trace(go.Scatter(
        x=[mid_E], y=[pos_N[k] - pad * 0.5], mode='text',
        text=[f'{dist:.0f} m'], textfont=dict(size=9, color='gray'),
        showlegend=False
    ), row=1, col=1)

figF.add_trace(go.Scatter(
    x=[(pos_E.min() + pos_E.max()) / 2],
    y=[pos_N.max() + pad * 1.5],
    mode='text',
    text=['↑ <b>Bắc</b>'], textfont=dict(size=13, color='darkgreen'),
    showlegend=False
), row=1, col=1)

figF.update_xaxes(
    title_text='← Tây      Đông (m) →', gridcolor='lightgray',
    zeroline=True, zerolinewidth=1.5, zerolinecolor='gray',
    range=[pos_E.min() - pad*2, pos_E.max() + pad*2], row=1, col=1
)
figF.update_yaxes(
    title_text='Bắc (m) ↑', gridcolor='lightgray',
    zeroline=True, zerolinewidth=1.5, zerolinecolor='gray',
    scaleanchor='x', scaleratio=1,
    range=[pos_N.min() - pad*3, pos_N.max() + pad*3], row=1, col=1
)

for b in range(NB):
    col_b = colors[b % len(colors)]
    for sign, sym in [(+1, 'circle'), (-1, 'x')]:
        figF.add_trace(go.Scatter(
            x=sign * u[:, b], y=sign * v[:, b],
            mode='markers', marker=dict(size=3, color=col_b, symbol=sym),
            name=bl_labels[b], showlegend=False, legendgroup=f'bl{b}',
            hovertemplate='u=%{x:.1f} λ  v=%{y:.1f} λ<extra>' + bl_labels[b] + '</extra>'
        ), row=1, col=2)

figF.update_xaxes(title_text='u (λ)', gridcolor='lightgray',
                  zeroline=True, zerolinewidth=1, zerolinecolor='gray', row=1, col=2)
figF.update_yaxes(title_text='v (λ)', gridcolor='lightgray',
                  scaleanchor='x2', scaleratio=1,
                  zeroline=True, zerolinewidth=1, zerolinecolor='gray', row=1, col=2)

figF.update_layout(
    height=640, width=1200,
    title_text=(f'BỐ TRÍ ANTEN (mét) & U-V COVERAGE — {ARRAY.name}  '
                f'(f={obs.fc_hz/1e6:.0f} MHz, lat={obs.lat_deg}°N)'),
    title_x=0.5, template='plotly_white',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center')
)
figF.show()

print(f"\nCell F Xong!")
print(f"  λ = {lam_m*100:.2f} cm tại {obs.fc_hz/1e6:.0f} MHz")
print(f"  Cửa sổ quan sát: H = ±{abs(H_arr).max():.1f}° = ±{abs(H_arr).max()/15:.2f}h → tổng {H_span_h:.1f} giờ")
print(f"  u: [{u.min():.1f}, {u.max():.1f}] λ  =  [{u.min()*lam_m:.2f}, {u.max()*lam_m:.2f}] m")
print(f"  v: [{v.min():.1f}, {v.max():.1f}] λ  =  [{v.min()*lam_m:.2f}, {v.max()*lam_m:.2f}] m")
print(f"  dec={obs.dec_deg:.1f}°  lat={obs.lat_deg}°")


[Validate] Dump #500 (H=0.092°) vs tham chiếu compute_uvw(H=0):
  Ant 0-1 (8m)           | u= -16.278 (ref= -16.278)  v=-0.000 (ref=-0.000)  OK
  Ant 0-2 (16m)          | u= -32.556 (ref= -32.556)  v=-0.000 (ref=-0.000)  OK
  Ant 0-3 (24m)          | u= -48.834 (ref= -48.834)  v=-0.000 (ref=-0.000)  OK
  Ant 1-2 (8m)           | u= -16.278 (ref= -16.278)  v=-0.000 (ref=-0.000)  OK
  Ant 1-3 (16m)          | u= -32.556 (ref= -32.556)  v=-0.000 (ref=-0.000)  OK
  Ant 2-3 (8m)           | u= -16.278 (ref= -16.278)  v=-0.000 (ref=-0.000)  OK



Cell F Xong!
  λ = 49.15 cm tại 610 MHz
  Cửa sổ quan sát: H = ±89.9° = ±5.99h → tổng 12.0 giờ
  u: [-48.8, -0.0] λ  =  [-24.00, -0.01] m
  v: [-0.0, -0.0] λ  =  [-0.00, -0.00] m
  dec=0.0°  lat=-22.22°


In [12]:
# %% CELL G: DIRTY IMAGE + PSF + PROFILE CẮT E-W & N-S
# Chứng minh: mảng Đông-Tây thuần tuý → không có độ phân giải hướng Bắc-Nam
import interfsim as ix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

FOV_DEG  = 3.0
GRID     = 256

# ── Dirty image (fringe-stop ON) ──────────────────────────────────────────
img, l_deg, m_deg = ix.make_dirty_image(
    u, v, w, visibilities,
    fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True,
)

# ── PSF / Dirty Beam (tất cả visibility = 1) ─────────────────────────────
psf, _, _ = ix.make_dirty_image(
    u, v, w, np.ones_like(visibilities),
    fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True,
)
psf = psf / psf.max()    # chuẩn hoá đỉnh = 1

l_arcmin = l_deg * 60    # arcmin
m_arcmin = m_deg * 60

# ── [Validate] Đỉnh dirty image ──────────────────────────────────────────
peak_idx      = np.unravel_index(np.argmax(img), img.shape)
peak_l_arcmin = l_arcmin[peak_idx[1]]
peak_m_arcmin = m_arcmin[peak_idx[0]]
ok = abs(peak_l_arcmin) < 3 and abs(peak_m_arcmin) < 3
print(f"[Validate] Peak tại (l,m) = ({peak_l_arcmin:.2f}', {peak_m_arcmin:.2f}') arcmin")
print("           " + ("OK: hội tụ vào tâm" if ok else "WARN: đỉnh lệch — kiểm tra fringe stop"))

# ── Cắt 1D qua tâm ───────────────────────────────────────────────────────
ic = np.argmin(np.abs(l_arcmin))   # cột gần l=0 nhất → trục N-S
ir = np.argmin(np.abs(m_arcmin))   # hàng gần m=0 nhất → trục E-W

ew_img = img[ir, :]    # hướng Đông-Tây
ns_img = img[:, ic]    # hướng Bắc-Nam
ew_psf = psf[ir, :]
ns_psf = psf[:, ic]

# ── FWHM ─────────────────────────────────────────────────────────────────
def fwhm_arcmin(profile, axis):
    half  = profile.max() / 2.0
    above = axis[profile >= half]
    return (above[-1] - above[0]) if len(above) >= 2 else float('nan')

fwhm_ew = fwhm_arcmin(ew_psf, l_arcmin)
fwhm_ns = fwhm_arcmin(ns_psf, m_arcmin)
ratio   = fwhm_ns / fwhm_ew
print(f"[FWHM PSF] Đông-Tây = {fwhm_ew:.1f}'   Bắc-Nam = {fwhm_ns:.1f}'")
print(f"  → Bắc-Nam kém hơn E-W  {ratio:.1f}×  (mảng Đông-Tây thuần tuý)")

# ── Figure 2×2 ───────────────────────────────────────────────────────────
sun_r   = 16.0
theta_c = np.linspace(0, 2 * np.pi, 300)
fov_lim = FOV_DEG * 30          # arcmin, nửa FOV

figG = make_subplots(
    rows=2, cols=2,
    vertical_spacing=0.14, horizontal_spacing=0.10,
    subplot_titles=(
        f"Dirty Image — {ARRAY.name}",
        f"PSF / Dirty Beam  (FWHM E-W={fwhm_ew:.1f}', N-S={fwhm_ns:.1f}')",
        "Cắt qua tâm — hướng Đông-Tây (l)",
        "Cắt qua tâm — hướng Bắc-Nam (m)  ← không có fringe",
    )
)

# ── Panel 1: Dirty image ──────────────────────────────────────────────────
figG.add_trace(go.Heatmap(
    z=img, x=l_arcmin, y=m_arcmin,
    colorscale='Hot',
    colorbar=dict(title="Cường độ", x=0.44, len=0.46, thickness=12),
    zmin=0, zmax=1
), row=1, col=1)
figG.add_trace(go.Scatter(
    x=sun_r * np.cos(theta_c), y=sun_r * np.sin(theta_c),
    mode='lines', line=dict(color='deepskyblue', width=2, dash='dash'),
    name="Đĩa MT (~32')"
), row=1, col=1)
figG.add_trace(go.Scatter(
    x=[peak_l_arcmin], y=[peak_m_arcmin], mode='markers+text',
    marker=dict(color='lime', size=10, symbol='cross'),
    text=[f"Peak ({peak_l_arcmin:.1f}', {peak_m_arcmin:.1f}')"],
    textposition='top right', textfont=dict(color='lime', size=9),
    showlegend=False
), row=1, col=1)

# ── Panel 2: PSF ──────────────────────────────────────────────────────────
figG.add_trace(go.Heatmap(
    z=psf, x=l_arcmin, y=m_arcmin,
    colorscale='RdBu_r',
    colorbar=dict(title="PSF", x=1.01, len=0.46, thickness=12),
    zmin=-0.5, zmax=1
), row=1, col=2)
# Đường FWHM E-W (vàng)
figG.add_shape(type='line',
    x0=-fwhm_ew/2, x1=fwhm_ew/2, y0=0, y1=0,
    line=dict(color='yellow', width=2.5), row=1, col=2)
# Đường FWHM N-S (cyan)
figG.add_shape(type='line',
    x0=0, x1=0, y0=-fwhm_ns/2, y1=fwhm_ns/2,
    line=dict(color='cyan', width=2.5), row=1, col=2)
# Nhãn FWHM
figG.add_trace(go.Scatter(
    x=[fwhm_ew/2 + 4], y=[2],
    mode='text', text=[f"←{fwhm_ew:.1f}'→"],
    textfont=dict(color='yellow', size=9), showlegend=False
), row=1, col=2)
figG.add_trace(go.Scatter(
    x=[4], y=[fwhm_ns/2 + 4],
    mode='text', text=[f"↕ {fwhm_ns:.1f}'"],
    textfont=dict(color='cyan', size=9), showlegend=False
), row=1, col=2)

# ── Panel 3: Cắt E-W ──────────────────────────────────────────────────────
figG.add_trace(go.Scatter(
    x=l_arcmin, y=ew_img,
    mode='lines', name='Dirty image',
    line=dict(color='darkorange', width=2),
    fill='tozeroy', fillcolor='rgba(255,140,0,0.15)',
    showlegend=False
), row=2, col=1)
figG.add_trace(go.Scatter(
    x=l_arcmin, y=ew_psf,
    mode='lines', name=f"PSF  FWHM={fwhm_ew:.1f}'",
    line=dict(color='white', width=1.5, dash='dash'),
    showlegend=False
), row=2, col=1)
figG.add_hline(y=0.5, line=dict(color='yellow', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ew:.1f}'",
               annotation_font_color='yellow', annotation_font_size=10,
               row=2, col=1)

# ── Panel 4: Cắt N-S ──────────────────────────────────────────────────────
figG.add_trace(go.Scatter(
    x=m_arcmin, y=ns_img,
    mode='lines', name='Dirty image N-S',
    line=dict(color='royalblue', width=2),
    fill='tozeroy', fillcolor='rgba(65,105,225,0.15)',
    showlegend=False
), row=2, col=2)
figG.add_trace(go.Scatter(
    x=m_arcmin, y=ns_psf,
    mode='lines', name=f"PSF N-S  FWHM={fwhm_ns:.1f}'",
    line=dict(color='white', width=1.5, dash='dash'),
    showlegend=False
), row=2, col=2)
figG.add_hline(y=0.5, line=dict(color='cyan', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ns:.1f}'",
               annotation_font_color='cyan', annotation_font_size=10,
               row=2, col=2)

# ── Axes ──────────────────────────────────────────────────────────────────
for col in (1, 2):
    figG.update_xaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                      title_text="Lệch Đông-Tây (arcmin)", row=1, col=col)
    figG.update_yaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                      title_text="Lệch Bắc-Nam (arcmin)", row=1, col=col)

figG.update_xaxes(title_text="l — Đông-Tây (arcmin)", gridcolor='rgba(128,128,128,0.3)', row=2, col=1)
figG.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=2, col=1)
figG.update_xaxes(title_text="m — Bắc-Nam (arcmin)",  gridcolor='rgba(128,128,128,0.3)', row=2, col=2)
figG.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=2, col=2)

figG.update_layout(
    title=dict(
        text=(
            f"DIRTY IMAGE & PSF — {ARRAY.name}  610 MHz  (fringe-stop ON)<br>"
            f"<sup>"
            f"Mảng Đông-Tây: FWHM E-W = {fwhm_ew:.1f}'  ←→  FWHM N-S = {fwhm_ns:.1f}'  "
            f"→ độ phân giải Bắc-Nam kém hơn E-W  {ratio:.1f}×  "
            f"vì không có baseline theo hướng N-S"
            f"</sup>"
        ),
        x=0.5, font=dict(size=13)
    ),
    height=1050, width=1100,
    template="plotly_dark",
    showlegend=True,
    legend=dict(x=0.01, y=0.99, font=dict(color='white', size=10))
)
figG.show()

print(f"\nCell G Xong!  Image {img.shape}  |  dynamic range: {img.max()/img.std():.1f}")
print(f"  FWHM E-W = {fwhm_ew:.1f}'  |  FWHM N-S = {fwhm_ns:.1f}'  |  tỉ lệ = {ratio:.1f}×")


[Validate] Peak tại (l,m) = (0.35', -90.00') arcmin
           WARN: đỉnh lệch — kiểm tra fringe stop
[FWHM PSF] Đông-Tây = 180.0'   Bắc-Nam = 180.0'
  → Bắc-Nam kém hơn E-W  1.0×  (mảng Đông-Tây thuần tuý)



Cell G Xong!  Image (256, 256)  |  dynamic range: 2.5
  FWHM E-W = 180.0'  |  FWHM N-S = 180.0'  |  tỉ lệ = 1.0×


In [13]:
# %% CELL G2: CASE 1.5 — CROSS 4 ANTEN (E-W + N-S) — DỮ LIỆU THỰC TỪ CELL A
# So sánh trực tiếp với Case 1: cùng 4 anten, cùng spacing, chỉ đổi bố trí
# → chứng minh bố trí 2D lấy lại được độ phân giải hướng Bắc-Nam
import interfsim as ix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ── 1. BỐ TRÍ CROSS: 2 anten E-W + 2 anten N-S, spacing lấy từ Case 1 ────
s = float(ARRAY.baseline_lengths.min())  # spacing từ Case 1 (8 m)

#        Ant3(0,+s)
#           |
# Ant0(-s,0)+------Ant1(+s,0)
#           |
#        Ant2(0,-s)

ARRAY_CROSS = ix.from_enu(
    [[-s, 0], [s, 0], [0, -s], [0, s]],
    name=f"cross_4ant_{s:.0f}m"
)
print(ARRAY_CROSS.summary())
print("Pairs            :", ARRAY_CROSS.pairs)
print("Baseline (m)     :", np.round(ARRAY_CROSS.baseline_lengths, 1))

# ── 2. SINH ĐIỆN ÁP — tái dùng obs, amp_env, noise_sigma, source_process ──
V_cross = ix.simulate_voltages(
    ARRAY_CROSS, obs,
    amplitude_envelope = amp_env,
    noise_sigma        = noise_sigma,
    source_process     = source_process,  # cùng nguồn coherent → so sánh công bằng
)

# ── 3. CORRELATOR ─────────────────────────────────────────────────────────
vis_cross, cen_cross = ix.correlate(V_cross, ARRAY_CROSS, INTEGRATION_BINS)
NB_c = vis_cross.shape[1]
bl_labels_c = [f"Ant {i}-{j} ({ARRAY_CROSS.baseline_lengths[b]:.0f}m)"
               for b, (i, j) in enumerate(ARRAY_CROSS.pairs)]

# ── 4. U-V + DIRTY IMAGE + PSF ────────────────────────────────────────────
obs_c = ix.Observation(obs.lat_deg, obs.dec_deg, obs.fc_hz,
                       H_deg=obs.H_deg[cen_cross])
u_c, v_c, w_c = ix.compute_uvw(ARRAY_CROSS.baseline_vectors_enu, obs_c)

FOV_DEG = 3.0
GRID    = 256
fov_lim = FOV_DEG * 30   # arcmin

img_c, l_c, m_c = ix.make_dirty_image(
    u_c, v_c, w_c, vis_cross, fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True)
psf_c, _,   _   = ix.make_dirty_image(
    u_c, v_c, w_c, np.ones_like(vis_cross), fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True)
psf_c /= psf_c.max()

# Tái tính PSF Case 1 để so sánh (dùng u,v,w từ Cell F)
psf_1, _, _ = ix.make_dirty_image(
    u, v, w, np.ones_like(visibilities), fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True)
psf_1 /= psf_1.max()

la = l_c * 60;  ma = m_c * 60   # arcmin

# ── 5. FWHM ───────────────────────────────────────────────────────────────
def fwhm_arcmin(profile, axis):
    above = axis[profile >= profile.max() / 2.0]
    return (above[-1] - above[0]) if len(above) >= 2 else float('nan')

ic_c = np.argmin(np.abs(la));  ir_c = np.argmin(np.abs(ma))
fwhm_ew_c = fwhm_arcmin(psf_c[ir_c, :], la)
fwhm_ns_c = fwhm_arcmin(psf_c[:, ic_c], ma)

# PSF Case 1
la_1 = l_c * 60;  ma_1 = m_c * 60
ic_1 = np.argmin(np.abs(la_1));  ir_1 = np.argmin(np.abs(ma_1))
fwhm_ew_1 = fwhm_arcmin(psf_1[ir_1, :], la_1)
fwhm_ns_1 = fwhm_arcmin(psf_1[:, ic_1], ma_1)

print(f"\n{'Layout':<22} {'FWHM E-W':>10} {'FWHM N-S':>10} {'Tỉ lệ NS/EW':>13}")
print(f"{'Case 1 (E-W thẳng)':<22} {fwhm_ew_1:>9.1f}' {fwhm_ns_1:>9.1f}' {fwhm_ns_1/fwhm_ew_1:>12.1f}×")
print(f"{'Case 1.5 (Cross 2D)':<22} {fwhm_ew_c:>9.1f}' {fwhm_ns_c:>9.1f}' {fwhm_ns_c/fwhm_ew_c:>12.2f}×")
print(f"\n→ Cross cải thiện độ phân giải N-S {fwhm_ns_1/fwhm_ns_c:.1f}× (từ {fwhm_ns_1:.1f}' xuống {fwhm_ns_c:.1f}')")

# ── 6. FIGURE 3×2 ─────────────────────────────────────────────────────────
colors_c = ['royalblue','darkorange','green','red','purple','brown']
sun_r    = 16.0
theta_p  = np.linspace(0, 2*np.pi, 300)
pos_E_c  = ARRAY_CROSS.positions_enu[:, 0]
pos_N_c  = ARRAY_CROSS.positions_enu[:, 1]

figG2 = make_subplots(
    rows=3, cols=2,
    vertical_spacing=0.09, horizontal_spacing=0.10,
    subplot_titles=(
        f"Bố trí anten — Cross 4 anten (spacing = {s:.0f} m)",
        "U-V Coverage — Cross 4 anten",
        f"Dirty Image — {ARRAY_CROSS.name}",
        f"PSF   FWHM E-W={fwhm_ew_c:.1f}'  N-S={fwhm_ns_c:.1f}'",
        "Cắt E-W qua tâm — có độ phân giải",
        "Cắt N-S qua tâm — phục hồi độ phân giải",
    )
)

# ── Hàng 1 trái: bố trí anten ──────────────────────────────────────────────
for b, (i, j) in enumerate(ARRAY_CROSS.pairs):
    figG2.add_trace(go.Scatter(
        x=[pos_E_c[i], pos_E_c[j]], y=[pos_N_c[i], pos_N_c[j]],
        mode='lines', line=dict(color=colors_c[b], width=2.5),
        name=bl_labels_c[b]
    ), row=1, col=1)
for k in range(ARRAY_CROSS.n_ant):
    figG2.add_trace(go.Scatter(
        x=[pos_E_c[k]], y=[pos_N_c[k]], mode='markers+text',
        marker=dict(size=20, color='white', symbol='square',
                    line=dict(color='navy', width=2)),
        text=[f'Ant {k}'], textposition='top center',
        textfont=dict(size=10, color='navy'), showlegend=False
    ), row=1, col=1)
figG2.add_trace(go.Scatter(
    x=[0], y=[s * 1.6], mode='text',
    text=['↑ Bắc'], textfont=dict(size=12, color='darkgreen'),
    showlegend=False
), row=1, col=1)
figG2.update_xaxes(title_text='← Tây   Đông (m) →', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray',
                   range=[-s*1.8, s*1.8], row=1, col=1)
figG2.update_yaxes(title_text='Bắc (m) ↑', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray',
                   range=[-s*1.8, s*1.8], row=1, col=1)

# ── Hàng 1 phải: u-v coverage ──────────────────────────────────────────────
for b in range(NB_c):
    for sign in [+1, -1]:
        figG2.add_trace(go.Scatter(
            x=sign * u_c[:, b], y=sign * v_c[:, b],
            mode='markers', marker=dict(size=2, color=colors_c[b % len(colors_c)]),
            showlegend=False
        ), row=1, col=2)
figG2.update_xaxes(title_text='u (λ)', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray', row=1, col=2)
figG2.update_yaxes(title_text='v (λ)', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray', row=1, col=2)

# ── Hàng 2: dirty image + PSF ──────────────────────────────────────────────
peak_c = np.unravel_index(np.argmax(img_c), img_c.shape)
pk_l   = la[peak_c[1]];  pk_m = ma[peak_c[0]]

figG2.add_trace(go.Heatmap(
    z=img_c, x=la, y=ma, colorscale='Hot',
    colorbar=dict(title="Cường độ", x=0.44, len=0.28, thickness=11, y=0.49),
    zmin=0, zmax=1
), row=2, col=1)
figG2.add_trace(go.Scatter(
    x=sun_r*np.cos(theta_p), y=sun_r*np.sin(theta_p),
    mode='lines', line=dict(color='deepskyblue', width=2, dash='dash'),
    showlegend=False
), row=2, col=1)
figG2.add_trace(go.Scatter(
    x=[pk_l], y=[pk_m], mode='markers',
    marker=dict(color='lime', size=10, symbol='cross'), showlegend=False
), row=2, col=1)

figG2.add_trace(go.Heatmap(
    z=psf_c, x=la, y=ma, colorscale='RdBu_r',
    colorbar=dict(title="PSF", x=1.01, len=0.28, thickness=11, y=0.49),
    zmin=-0.5, zmax=1
), row=2, col=2)
figG2.add_shape(type='line', x0=-fwhm_ew_c/2, x1=fwhm_ew_c/2, y0=0, y1=0,
                line=dict(color='yellow', width=2.5), row=2, col=2)
figG2.add_shape(type='line', x0=0, x1=0, y0=-fwhm_ns_c/2, y1=fwhm_ns_c/2,
                line=dict(color='cyan', width=2.5), row=2, col=2)

for col in (1, 2):
    figG2.update_xaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                       title_text="Lệch Đông-Tây (arcmin)", row=2, col=col)
    figG2.update_yaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                       title_text="Lệch Bắc-Nam (arcmin)", row=2, col=col)

# ── Hàng 3: profile 1D ─────────────────────────────────────────────────────
ew_img_c = img_c[ir_c, :];  ns_img_c = img_c[:, ic_c]
ew_psf_c = psf_c[ir_c, :];  ns_psf_c = psf_c[:, ic_c]

figG2.add_trace(go.Scatter(
    x=la, y=ew_img_c, mode='lines',
    line=dict(color='darkorange', width=2),
    fill='tozeroy', fillcolor='rgba(255,140,0,0.15)', showlegend=False
), row=3, col=1)
figG2.add_trace(go.Scatter(
    x=la, y=ew_psf_c, mode='lines',
    line=dict(color='white', width=1.5, dash='dash'), showlegend=False
), row=3, col=1)
figG2.add_hline(y=0.5, line=dict(color='yellow', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ew_c:.1f}'",
               annotation_font_color='yellow', annotation_font_size=10, row=3, col=1)

figG2.add_trace(go.Scatter(
    x=ma, y=ns_img_c, mode='lines',
    line=dict(color='royalblue', width=2),
    fill='tozeroy', fillcolor='rgba(65,105,225,0.15)', showlegend=False
), row=3, col=2)
figG2.add_trace(go.Scatter(
    x=ma, y=ns_psf_c, mode='lines',
    line=dict(color='white', width=1.5, dash='dash'), showlegend=False
), row=3, col=2)
figG2.add_hline(y=0.5, line=dict(color='cyan', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ns_c:.1f}'",
               annotation_font_color='cyan', annotation_font_size=10, row=3, col=2)

figG2.update_xaxes(title_text="l — Đông-Tây (arcmin)", gridcolor='rgba(128,128,128,0.3)', row=3, col=1)
figG2.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=3, col=1)
figG2.update_xaxes(title_text="m — Bắc-Nam (arcmin)",  gridcolor='rgba(128,128,128,0.3)', row=3, col=2)
figG2.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=3, col=2)

figG2.update_layout(
    title=dict(
        text=(
            f"CASE 1.5 — CROSS 4 ANTEN (spacing={s:.0f} m, dữ liệu thực)  ←→  Case 1 (E-W thẳng hàng)<br>"
            f"<sup>"
            f"Case 1 E-W: FWHM E-W={fwhm_ew_1:.1f}', N-S={fwhm_ns_1:.1f}' (tỉ lệ {fwhm_ns_1/fwhm_ew_1:.1f}×)  ·  "
            f"Cross 2D: FWHM E-W={fwhm_ew_c:.1f}', N-S={fwhm_ns_c:.1f}' (tỉ lệ {fwhm_ns_c/fwhm_ew_c:.2f}×)  ·  "
            f"N-S cải thiện {fwhm_ns_1/fwhm_ns_c:.1f}× khi thêm baseline N-S"
            f"</sup>"
        ),
        x=0.5, font=dict(size=13)
    ),
    height=1400, width=1100,
    template="plotly_dark",
    showlegend=True,
    legend=dict(x=0.01, y=0.99, font=dict(color='white', size=9),
                bgcolor='rgba(0,0,0,0.5)', bordercolor='gray', borderwidth=1)
)
figG2.show()

print(f"\nCell G2 Xong!")
print(f"  Cross 4-ant: FWHM E-W={fwhm_ew_c:.1f}'  N-S={fwhm_ns_c:.1f}'  tỉ lệ={fwhm_ns_c/fwhm_ew_c:.2f}×")
print(f"  Case 1 E-W:  FWHM E-W={fwhm_ew_1:.1f}'  N-S={fwhm_ns_1:.1f}'  tỉ lệ={fwhm_ns_1/fwhm_ew_1:.1f}×")
print(f"  → Bố trí 2D phục hồi độ phân giải N-S {fwhm_ns_1/fwhm_ns_c:.1f}× so với Case 1")


cross_4ant_8m: N=4 anten, 6 baseline, baseline min/max = 11.3/16.0 m
Pairs            : [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
Baseline (m)     : [16.  11.3 11.3 11.3 11.3 16. ]

Layout                   FWHM E-W   FWHM N-S   Tỉ lệ NS/EW
Case 1 (E-W thẳng)         180.0'     180.0'          1.0×
Case 1.5 (Cross 2D)          nan'      62.1'          nan×

→ Cross cải thiện độ phân giải N-S 2.9× (từ 180.0' xuống 62.1')



Cell G2 Xong!
  Cross 4-ant: FWHM E-W=nan'  N-S=62.1'  tỉ lệ=nan×
  Case 1 E-W:  FWHM E-W=180.0'  N-S=180.0'  tỉ lệ=1.0×
  → Bố trí 2D phục hồi độ phân giải N-S 2.9× so với Case 1


In [14]:
# %% CELL G3: CASE 1.75 — TAM GIÁC + TÂM KIỂU DLITE — DỮ LIỆU THỰC TỪ CELL A
# Nguồn: Helmboldt et al. 2021, "The Deployable Low-Band Ionosphere and
# Transient Experiment (DLITE)", Radio Science 56, e2021RS007298,
# DOI: 10.1029/2021RS007298 — mảng DLITE thực tế dùng 4 anten LWA tái sử dụng,
# bố trí 3 anten ở đỉnh tam giác đều (baseline ~200–500 m ngoài đời) + 1 anten
# ở tâm, KHÔNG PHẢI hình chữ thập/vuông. Lý do: với layout chữ thập (Case 1.5),
# 2 cặp baseline đối xứng song song nhau → track u-v của chúng trùng lên nhau
# khi Trái Đất quay (dư thừa). Tam giác đều + tâm không có baseline nào song
# song → cả 6 vector baseline quét 6 elip u-v khác phương, tối đa hoá coverage
# không dư thừa cho đúng 4 anten — đây là lý do DLITE chọn layout này.
# So sánh trực tiếp với Case 1 (E-W) và Case 1.5 (Cross): cùng 4 anten, cùng
# spacing gốc s, chỉ đổi bố trí hình học.
import interfsim as ix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ── 1. BỐ TRÍ TAM GIÁC + TÂM: cạnh tam giác = s (spacing từ Case 1) ───────
#    (Case 1.5 dùng s làm bán-cạnh cross; ở đây dùng s làm CẠNH tam giác để
#     baseline dài nhất của layout mới ≈ baseline dài nhất Case 1.5 = 2s? Không:
#     ta giữ s là "đơn vị chiều dài gốc" chung cho mọi case, đúng quy ước đã
#     dùng ở Cell G2, không cố ép bằng baseline min/max.)
#
#            Ant1 (Bắc)
#             / \
#            /   \
#      Ant3 •──Ant0──• Ant2      Ant0 = anten TÂM (giữa)
#         (Tây-Nam) (Đông-Nam)
ARRAY_TRI = ix.preset_dlite(side=s)
print(ARRAY_TRI.summary())
print("Pairs            :", ARRAY_TRI.pairs)
print("Baseline (m)     :", np.round(ARRAY_TRI.baseline_lengths, 1))


def n_unique_baseline_directions(config, decimals=1):
    """Đếm số PHƯƠNG baseline khác nhau (mod 180°) — đo độ dư thừa u-v.
    Layout càng ít phương trùng nhau thì u-v coverage càng ít chồng lấp."""
    vecs = config.baseline_vectors_enu[:, :2]
    ang = np.round(np.degrees(np.arctan2(vecs[:, 1], vecs[:, 0])) % 180.0, decimals)
    return len(np.unique(ang))


n_dir_cross = n_unique_baseline_directions(ARRAY_CROSS)
n_dir_tri = n_unique_baseline_directions(ARRAY_TRI)
print(f"\nSố phương baseline khác nhau (trên 6 baseline):")
print(f"  Case 1.5 (Cross)      : {n_dir_cross}/6  (baseline đối xứng → trùng phương)")
print(f"  Case 1.75 (Tam giác)  : {n_dir_tri}/6  (không baseline nào song song)")

# ── 2. SINH ĐIỆN ÁP — tái dùng obs, amp_env, noise_sigma, source_process ──
V_tri = ix.simulate_voltages(
    ARRAY_TRI, obs,
    amplitude_envelope = amp_env,
    noise_sigma        = noise_sigma,
    source_process     = source_process,  # cùng nguồn coherent → so sánh công bằng
)

# ── 3. CORRELATOR ─────────────────────────────────────────────────────────
vis_tri, cen_tri = ix.correlate(V_tri, ARRAY_TRI, INTEGRATION_BINS)
NB_t = vis_tri.shape[1]
bl_labels_t = [f"Ant {i}-{j} ({ARRAY_TRI.baseline_lengths[b]:.1f}m)"
               for b, (i, j) in enumerate(ARRAY_TRI.pairs)]

# ── 4. U-V + DIRTY IMAGE + PSF ────────────────────────────────────────────
obs_t = ix.Observation(obs.lat_deg, obs.dec_deg, obs.fc_hz,
                       H_deg=obs.H_deg[cen_tri])
u_t, v_t, w_t = ix.compute_uvw(ARRAY_TRI.baseline_vectors_enu, obs_t)

img_t, l_t, m_t = ix.make_dirty_image(
    u_t, v_t, w_t, vis_tri, fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True)
psf_t, _,   _   = ix.make_dirty_image(
    u_t, v_t, w_t, np.ones_like(vis_tri), fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=True)
psf_t /= psf_t.max()

la_t = l_t * 60;  ma_t = m_t * 60   # arcmin

# ── 5. FWHM ───────────────────────────────────────────────────────────────
ic_t = np.argmin(np.abs(la_t));  ir_t = np.argmin(np.abs(ma_t))
fwhm_ew_t = fwhm_arcmin(psf_t[ir_t, :], la_t)
fwhm_ns_t = fwhm_arcmin(psf_t[:, ic_t], ma_t)

print(f"\n{'Layout':<22} {'FWHM E-W':>10} {'FWHM N-S':>10} {'Tỉ lệ NS/EW':>13}")
print(f"{'Case 1 (E-W thẳng)':<22} {fwhm_ew_1:>9.1f}' {fwhm_ns_1:>9.1f}' {fwhm_ns_1/fwhm_ew_1:>12.1f}×")
print(f"{'Case 1.5 (Cross 2D)':<22} {fwhm_ew_c:>9.1f}' {fwhm_ns_c:>9.1f}' {fwhm_ns_c/fwhm_ew_c:>12.2f}×")
print(f"{'Case 1.75 (Tam giác)':<22} {fwhm_ew_t:>9.1f}' {fwhm_ns_t:>9.1f}' {fwhm_ns_t/fwhm_ew_t:>12.2f}×")
print(f"\n→ Tam giác+tâm (kiểu DLITE) cải thiện N-S {fwhm_ns_1/fwhm_ns_t:.1f}× so với Case 1"
      f", và có {n_dir_tri}/6 phương baseline khác nhau so với {n_dir_cross}/6 của Cross"
      f" → u-v coverage đều hơn, ít lỗ hổng hơn.")

# ── 6. FIGURE 3×2 ─────────────────────────────────────────────────────────
colors_t = ['royalblue', 'darkorange', 'green', 'red', 'purple', 'brown']
sun_r    = 16.0
theta_p  = np.linspace(0, 2*np.pi, 300)
pos_E_t  = ARRAY_TRI.positions_enu[:, 0]
pos_N_t  = ARRAY_TRI.positions_enu[:, 1]
tri_lim  = s * 1.2

figG3 = make_subplots(
    rows=3, cols=2,
    vertical_spacing=0.09, horizontal_spacing=0.10,
    subplot_titles=(
        f"Bố trí anten — Tam giác đều + tâm kiểu DLITE (cạnh = {s:.0f} m)",
        "U-V Coverage — Tam giác + tâm",
        f"Dirty Image — {ARRAY_TRI.name}",
        f"PSF   FWHM E-W={fwhm_ew_t:.1f}'  N-S={fwhm_ns_t:.1f}'",
        "Cắt E-W qua tâm",
        "Cắt N-S qua tâm",
    )
)

# ── Hàng 1 trái: bố trí anten ──────────────────────────────────────────────
for b, (i, j) in enumerate(ARRAY_TRI.pairs):
    figG3.add_trace(go.Scatter(
        x=[pos_E_t[i], pos_E_t[j]], y=[pos_N_t[i], pos_N_t[j]],
        mode='lines', line=dict(color=colors_t[b], width=2.5),
        name=bl_labels_t[b]
    ), row=1, col=1)
for k in range(ARRAY_TRI.n_ant):
    figG3.add_trace(go.Scatter(
        x=[pos_E_t[k]], y=[pos_N_t[k]], mode='markers+text',
        marker=dict(size=20, color='white', symbol='square',
                    line=dict(color='navy', width=2)),
        text=[f'Ant {k}'], textposition='top center',
        textfont=dict(size=10, color='navy'), showlegend=False
    ), row=1, col=1)
figG3.add_trace(go.Scatter(
    x=[0], y=[tri_lim * 0.9], mode='text',
    text=['↑ Bắc'], textfont=dict(size=12, color='darkgreen'),
    showlegend=False
), row=1, col=1)
figG3.update_xaxes(title_text='← Tây   Đông (m) →', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray',
                   range=[-tri_lim, tri_lim], row=1, col=1)
figG3.update_yaxes(title_text='Bắc (m) ↑', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray',
                   range=[-tri_lim, tri_lim], row=1, col=1)

# ── Hàng 1 phải: u-v coverage ──────────────────────────────────────────────
for b in range(NB_t):
    for sign in [+1, -1]:
        figG3.add_trace(go.Scatter(
            x=sign * u_t[:, b], y=sign * v_t[:, b],
            mode='markers', marker=dict(size=2, color=colors_t[b % len(colors_t)]),
            showlegend=False
        ), row=1, col=2)
figG3.update_xaxes(title_text='u (λ)', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray', row=1, col=2)
figG3.update_yaxes(title_text='v (λ)', gridcolor='lightgray',
                   zeroline=True, zerolinecolor='gray', row=1, col=2)

# ── Hàng 2: dirty image + PSF ──────────────────────────────────────────────
peak_t = np.unravel_index(np.argmax(img_t), img_t.shape)
pk_l_t = la_t[peak_t[1]];  pk_m_t = ma_t[peak_t[0]]

figG3.add_trace(go.Heatmap(
    z=img_t, x=la_t, y=ma_t, colorscale='Hot',
    colorbar=dict(title="Cường độ", x=0.44, len=0.28, thickness=11, y=0.49),
    zmin=0, zmax=1
), row=2, col=1)
figG3.add_trace(go.Scatter(
    x=sun_r*np.cos(theta_p), y=sun_r*np.sin(theta_p),
    mode='lines', line=dict(color='deepskyblue', width=2, dash='dash'),
    showlegend=False
), row=2, col=1)
figG3.add_trace(go.Scatter(
    x=[pk_l_t], y=[pk_m_t], mode='markers',
    marker=dict(color='lime', size=10, symbol='cross'), showlegend=False
), row=2, col=1)

figG3.add_trace(go.Heatmap(
    z=psf_t, x=la_t, y=ma_t, colorscale='RdBu_r',
    colorbar=dict(title="PSF", x=1.01, len=0.28, thickness=11, y=0.49),
    zmin=-0.5, zmax=1
), row=2, col=2)
figG3.add_shape(type='line', x0=-fwhm_ew_t/2, x1=fwhm_ew_t/2, y0=0, y1=0,
                line=dict(color='yellow', width=2.5), row=2, col=2)
figG3.add_shape(type='line', x0=0, x1=0, y0=-fwhm_ns_t/2, y1=fwhm_ns_t/2,
                line=dict(color='cyan', width=2.5), row=2, col=2)

for col in (1, 2):
    figG3.update_xaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                       title_text="Lệch Đông-Tây (arcmin)", row=2, col=col)
    figG3.update_yaxes(range=[-fov_lim, fov_lim], gridcolor='rgba(128,128,128,0.3)',
                       title_text="Lệch Bắc-Nam (arcmin)", row=2, col=col)

# ── Hàng 3: profile 1D ─────────────────────────────────────────────────────
ew_img_t = img_t[ir_t, :];  ns_img_t = img_t[:, ic_t]
ew_psf_t = psf_t[ir_t, :];  ns_psf_t = psf_t[:, ic_t]

figG3.add_trace(go.Scatter(
    x=la_t, y=ew_img_t, mode='lines',
    line=dict(color='darkorange', width=2),
    fill='tozeroy', fillcolor='rgba(255,140,0,0.15)', showlegend=False
), row=3, col=1)
figG3.add_trace(go.Scatter(
    x=la_t, y=ew_psf_t, mode='lines',
    line=dict(color='white', width=1.5, dash='dash'), showlegend=False
), row=3, col=1)
figG3.add_hline(y=0.5, line=dict(color='yellow', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ew_t:.1f}'",
               annotation_font_color='yellow', annotation_font_size=10, row=3, col=1)

figG3.add_trace(go.Scatter(
    x=ma_t, y=ns_img_t, mode='lines',
    line=dict(color='royalblue', width=2),
    fill='tozeroy', fillcolor='rgba(65,105,225,0.15)', showlegend=False
), row=3, col=2)
figG3.add_trace(go.Scatter(
    x=ma_t, y=ns_psf_t, mode='lines',
    line=dict(color='white', width=1.5, dash='dash'), showlegend=False
), row=3, col=2)
figG3.add_hline(y=0.5, line=dict(color='cyan', width=1, dash='dot'),
               annotation_text=f"FWHM={fwhm_ns_t:.1f}'",
               annotation_font_color='cyan', annotation_font_size=10, row=3, col=2)

figG3.update_xaxes(title_text="l — Đông-Tây (arcmin)", gridcolor='rgba(128,128,128,0.3)', row=3, col=1)
figG3.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=3, col=1)
figG3.update_xaxes(title_text="m — Bắc-Nam (arcmin)",  gridcolor='rgba(128,128,128,0.3)', row=3, col=2)
figG3.update_yaxes(title_text="Cường độ chuẩn hoá",    gridcolor='rgba(128,128,128,0.3)', row=3, col=2)

figG3.update_layout(
    title=dict(
        text=(
            f"CASE 1.75 — TAM GIÁC ĐỀU + TÂM KIỂU DLITE (cạnh={s:.0f} m, dữ liệu thực)  "
            f"←→  Case 1 (E-W) & Case 1.5 (Cross)<br>"
            f"<sup>"
            f"Case 1 E-W: FWHM E-W={fwhm_ew_1:.1f}', N-S={fwhm_ns_1:.1f}' (tỉ lệ {fwhm_ns_1/fwhm_ew_1:.1f}×)  ·  "
            f"Cross: FWHM E-W={fwhm_ew_c:.1f}', N-S={fwhm_ns_c:.1f}' (tỉ lệ {fwhm_ns_c/fwhm_ew_c:.2f}×)  ·  "
            f"Tam giác: FWHM E-W={fwhm_ew_t:.1f}', N-S={fwhm_ns_t:.1f}' (tỉ lệ {fwhm_ns_t/fwhm_ew_t:.2f}×)  ·  "
            f"phương baseline khác nhau: {n_dir_tri}/6 (tam giác) so {n_dir_cross}/6 (cross)"
            f"</sup>"
        ),
        x=0.5, font=dict(size=13)
    ),
    height=1400, width=1100,
    template="plotly_dark",
    showlegend=True,
    legend=dict(x=0.01, y=0.99, font=dict(color='white', size=9),
                bgcolor='rgba(0,0,0,0.5)', bordercolor='gray', borderwidth=1)
)
figG3.show()

print(f"\nCell G3 Xong!")
print(f"  Tam giác+tâm: FWHM E-W={fwhm_ew_t:.1f}'  N-S={fwhm_ns_t:.1f}'  tỉ lệ={fwhm_ns_t/fwhm_ew_t:.2f}×")
print(f"  → So với Case 1: cải thiện N-S {fwhm_ns_1/fwhm_ns_t:.1f}×")
print(f"  → So với Case 1.5 (Cross): {n_dir_tri}/6 phương baseline khác nhau (Cross chỉ {n_dir_cross}/6)"
      f", u-v coverage ít dư thừa hơn — đúng lý do DLITE (Helmboldt+ 2021) chọn layout này.")

dlite_triangle_center_8m: N=4 anten, 6 baseline, baseline min/max = 4.6/8.0 m
Pairs            : [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
Baseline (m)     : [4.6 4.6 4.6 8.  8.  8. ]

Số phương baseline khác nhau (trên 6 baseline):
  Case 1.5 (Cross)      : 4/6  (baseline đối xứng → trùng phương)
  Case 1.75 (Tam giác)  : 6/6  (không baseline nào song song)

Layout                   FWHM E-W   FWHM N-S   Tỉ lệ NS/EW
Case 1 (E-W thẳng)         180.0'     180.0'          1.0×
Case 1.5 (Cross 2D)          nan'      62.1'          nan×
Case 1.75 (Tam giác)         nan'      19.8'          nan×

→ Tam giác+tâm (kiểu DLITE) cải thiện N-S 9.1× so với Case 1, và có 6/6 phương baseline khác nhau so với 4/6 của Cross → u-v coverage đều hơn, ít lỗ hổng hơn.



Cell G3 Xong!
  Tam giác+tâm: FWHM E-W=nan'  N-S=19.8'  tỉ lệ=nan×
  → So với Case 1: cải thiện N-S 9.1×
  → So với Case 1.5 (Cross): 6/6 phương baseline khác nhau (Cross chỉ 4/6), u-v coverage ít dư thừa hơn — đúng lý do DLITE (Helmboldt+ 2021) chọn layout này.


## CASE 2 — Mảng mở rộng 2D + Mô hình thuận (Forward Visibility)

In [15]:
# %% CELL H: CASE 2 — MẢNG MỞ RỘNG 2D + FORWARD MODEL (đĩa Mặt Trời)
import interfsim as ix
from math import radians
import plotly.graph_objects as go
import numpy as np
import os
import gc  # Thư viện ép giải phóng RAM

# 1. TẠO THƯ MỤC LƯU TRỮ TRÊN Ổ E
save_dir_H = os.path.join(IMG_DIR, "case2_forward_model")   # từ Cell 0
os.makedirs(save_dir_H, exist_ok=True)

ARRAY2 = ix.preset_extended()
print(ARRAY2.summary())

# Sky model: đĩa Mặt Trời đều (Van Cittert-Zernike, đường kính ~0.53°)
sun = {'type': 'disk', 'diameter_rad': radians(0.53)}

N_IMG   = 600
obs_img = ix.Observation(
    lat_deg = obs.lat_deg,
    dec_deg = obs.dec_deg,
    fc_hz   = obs.fc_hz,
    H_deg   = np.linspace(obs.H_deg[0], obs.H_deg[-1], N_IMG),
)
bz_img = BEAM.response_voltage_offset(ix.zenith_angle_deg(obs_img)) ** 2

# Tính toán nặng
u2, v2, w2, V2 = ix.forward_visibilities(
    ARRAY2, obs_img, sun, time_weight=bz_img,
)

img2, l2_deg, m2_deg = ix.make_dirty_image(
    u2, v2, w2, V2, fov_deg=2.2, grid=256, do_fringe_stop=False,
)

# 2. LƯU DỮ LIỆU & GIẢI PHÓNG RAM NGAY LẬP TỨC
file_path_H = os.path.join(save_dir_H, "case2_data.npz")
np.savez_compressed(file_path_H, u2=u2, v2=v2, w2=w2, V2=V2, img2=img2, l2_deg=l2_deg, m2_deg=m2_deg)
print(f"[LƯU TRỮ] Đã nén và lưu dữ liệu an toàn vào: {file_path_H}")

# Xóa các ma trận thô tỷ lệ thuận với số lượng baseline khỏi RAM
del u2, v2, w2, V2
gc.collect()  # Ép hệ thống nhả RAM ngay lập tức

# [Validate]
peak2 = np.unravel_index(np.argmax(img2), img2.shape)
pl2   = l2_deg[peak2[1]] * 60
pm2   = m2_deg[peak2[0]] * 60
ok2   = abs(pl2) < 1.5 and abs(pm2) < 1.5
print(f"[Validate] CASE 2 peak: ({pl2:.1f}', {pm2:.1f}')  "
      + ("OK" if ok2 else "WARN: đỉnh lệch"))

l2_arcmin = l2_deg * 60
m2_arcmin = m2_deg * 60
sun_r     = 16.0
theta_c   = np.linspace(0, 2 * np.pi, 300)

figH = go.Figure()
figH.add_trace(go.Heatmap(z=img2, x=l2_arcmin, y=m2_arcmin,
    colorscale='Hot', colorbar=dict(title="Cường độ"), zmin=0, zmax=1))
figH.add_trace(go.Scatter(
    x=sun_r * np.cos(theta_c), y=sun_r * np.sin(theta_c),
    mode='lines', line=dict(color='deepskyblue', width=2, dash='dash'),
    name="Đĩa Mặt Trời (~32')"))
figH.add_trace(go.Scatter(
    x=[pl2], y=[pm2], mode='markers+text',
    marker=dict(color='lime', size=10, symbol='cross'),
    text=[f"Peak ({pl2:.1f}', {pm2:.1f}')"],
    textposition='top right', textfont=dict(color='lime'), name='Peak'))

figH.update_layout(
    title=f"DIRTY IMAGE CASE 2 — {ARRAY2.name}  (forward model, N_IMG={N_IMG}, fringe-stop OFF)",
    xaxis_title="Lệch Đông-Tây (arcmin)",
    yaxis_title="Lệch Bắc-Nam (arcmin)",
    width=750, height=750, template="plotly_dark",
    legend=dict(x=0.01, y=0.99, font=dict(color='white')))
figH.update_yaxes(scaleanchor="x", scaleratio=1)
figH.show()

# Xóa nốt mảng ảnh sau khi vẽ xong
del img2
gc.collect()
print(f"\nCell H Xong! RAM đã được dọn dẹp.")

extended_graded_cross_2D: N=17 anten, 136 baseline, baseline min/max = 5.0/130.0 m
[LƯU TRỮ] Đã nén và lưu dữ liệu an toàn vào: E:\1A DATN REAL\Cell_I_Data\case2_forward_model\case2_data.npz
[Validate] CASE 2 peak: (0.3', -0.3')  OK



Cell H Xong! RAM đã được dọn dẹp.


In [16]:
# =====================================================================
#  CELL X  —  XUẤT ẢNH RIÊNG LẺ CHO BÁO CÁO (thesis figure export)
#
#  ĐẶT CELL NÀY: sau Cell H (Case 2), trước Cell I (Sweep)
#  LÝ DO: Cell I ghi đè u, v, w, img, psf trong vòng lặp sweep
#
#  Kết quả: các file .png trong  IMG_DIR/thesis_figures/
#  Bước tiếp: copy toàn bộ vào  d:\datn\figures\
# =====================================================================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.signal import fftconvolve
from scipy.special import j1
import os, shutil

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'figure.dpi': 150,
})

# ── Thư mục đầu ra ────────────────────────────────────────────────────
THESIS_FIG_DIR = os.path.join(IMG_DIR, "thesis_figures")
os.makedirs(THESIS_FIG_DIR, exist_ok=True)

def _save(name):
    p = os.path.join(THESIS_FIG_DIR, name)
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close('all')
    print(f"  [OK] {name}")

# Hằng dùng chung
theta_c = np.linspace(0, 2*np.pi, 300)
sun_r   = 16.0   # bán kính Mặt Trời ~ 16 arcmin

print("=" * 55)
print("Cell X: Bắt đầu xuất ảnh riêng lẻ cho báo cáo")
print("=" * 55)

# ══════════════════════════════════════════════════════════════════════
# [1]  fig_sws_flux_day
#      Thông lượng Learmonth 610 MHz theo thời gian
#      Nguồn: sfu_data (Cell A)
# ══════════════════════════════════════════════════════════════════════
t_h = np.linspace(0, 12, len(sfu_data))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_h, sfu_data, color='darkorange', lw=0.9)
ax.set_xlabel("Time from transit window start  (hours)")
ax.set_ylabel("Solar flux density  (SFU)")
ax.set_title("Learmonth Solar Observatory — 610 MHz flux, 20 March 2025")
ax.grid(True, alpha=0.3, ls='--')
fig.tight_layout()
_save("fig_sws_flux_day.png")

# ══════════════════════════════════════════════════════════════════════
# [2]  fig_voltage_env
#      Envelope RMS điện áp anten (bell curve + noise floor)
#      Nguồn: V_matrix (Cell C), V_noise, T_sys (Cell A / Cell 0)
#      NOTE: ch4_pipeline.tex đã được sửa sang {fig_voltage_env}
# ══════════════════════════════════════════════════════════════════════
WINDOW = 500
def _srms(x, w):
    x2 = np.abs(x)**2
    cs = np.concatenate([[0], np.cumsum(x2)])
    r  = np.sqrt((cs[w:] - cs[:-w]) / w)
    return np.concatenate([np.full(w-1, r[0]), r])

V_env_uV   = _srms(V_matrix[:, 0], WINDOW) * 1e6
V_noise_uV = V_noise * 1e6

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.fill_between(t_h, V_env_uV, alpha=0.18, color='royalblue')
ax.plot(t_h, V_env_uV, color='royalblue', lw=1.0,
        label='RMS envelope  (Ant 1, 500-sample window)')
ax.axhline(V_noise_uV, color='red', lw=1.3, ls='--',
           label=f'Johnson–Nyquist noise floor  ({V_noise_uV:.3f} µV,  '
                 f'T_sys = {T_sys} K)')
ax.set_xlabel("Time from transit window start  (hours)")
ax.set_ylabel("RMS voltage  (µV)")
ax.set_title("Per-antenna signal amplitude — 500-sample sliding RMS,  20 March 2025")
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3, ls='--')
fig.tight_layout()
_save("fig_voltage_env.png")

# ══════════════════════════════════════════════════════════════════════
# [3]  fig_fringe_pattern
#      Fringe thực sự: Re(V₀₁) theo thời gian → chu kỳ ~7.5 phút
#      Nguồn: visibilities, IDX_01 (Cell E)
# ══════════════════════════════════════════════════════════════════════
t_vis       = np.linspace(0, 12, NI)
fringe_real = np.real(visibilities[:, IDX_01])

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5),
                          gridspec_kw={'height_ratios': [1, 2], 'hspace': 0.42})

# Panel trên: toàn transit 12h
axes[0].plot(t_vis, fringe_real, color='royalblue', lw=0.5, alpha=0.8)
axes[0].set_ylabel("Re(V₀₁)  (V²)")
axes[0].set_title(f"Interference fringe — {bl_labels[IDX_01]},  "
                  f"20 March 2025  (δ = 0°, 12-h transit)")
axes[0].grid(True, alpha=0.3, ls='--')

# Panel dưới: zoom 2h quanh transit để thấy rõ chu kỳ
mask  = (t_vis >= 5.0) & (t_vis <= 7.0)
t_min = (t_vis[mask] - 6.0) * 60   # phút lệch so với đỉnh transit

axes[1].plot(t_min, fringe_real[mask], color='royalblue', lw=1.0)
axes[1].set_xlabel("Time relative to transit peak  (minutes)")
axes[1].set_ylabel("Re(V₀₁)  (V²)")
axes[1].set_title("Zoom: ±60 min around transit  →  fringe period ≈ 7.5 min")
axes[1].grid(True, alpha=0.3, ls='--')

fig.tight_layout()
_save("fig_fringe_pattern.png")

# ══════════════════════════════════════════════════════════════════════
# [4]  fig_uv_coverage
#      UV plane của compact EW array (Case 1)
#      Nguồn: u, v (Cell F)  — phải chạy trước Cell I
# ══════════════════════════════════════════════════════════════════════
cmap_bl = plt.cm.tab10(np.linspace(0, 0.9, NB))

fig, ax = plt.subplots(figsize=(8.5, 4.0))
for b in range(NB):
    kw = dict(s=1.5, color=cmap_bl[b], label=bl_labels[b])
    ax.scatter( u[:, b],  v[:, b], **kw)
    ax.scatter(-u[:, b], -v[:, b], s=1.5, color=cmap_bl[b])
ax.set_xlabel("u  (wavelengths λ)")
ax.set_ylabel("v  (wavelengths λ)")
ax.set_title("UV coverage — compact E–W array (Case 1)\n"
             "20 March 2025,  H ∈ [±90°]  (δ = 0° → v ≡ 0,  EW array has no N–S resolution)")
_u_max = float(np.abs(u).max()) * 1.08
ax.set_xlim(-_u_max, _u_max)
ax.set_ylim(-_u_max * 0.12, _u_max * 0.12)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(markerscale=6, fontsize=8, loc='upper right')
fig.tight_layout()
_save("fig_uv_coverage.png")

# ══════════════════════════════════════════════════════════════════════
# [5]  fig_visibility_jinc
#      |V| vs baseline + jinc lý thuyết (Case 2, extended array)
#      Nguồn: case2_data.npz (Cell H)
# ══════════════════════════════════════════════════════════════════════
npz_path2 = os.path.join(IMG_DIR, "case2_forward_model", "case2_data.npz")
if os.path.exists(npz_path2):
    d2   = np.load(npz_path2)
    u2_  = d2['u2']; v2_ = d2['v2']; V2_ = d2['V2']
    NI2  = V2_.shape[0]; t2 = NI2 // 2
    bl2  = np.sqrt(u2_[t2]**2 + v2_[t2]**2)     # baseline (λ) tại transit
    va2  = np.abs(V2_[t2]); va2 = va2 / va2.max()

    theta_s = np.radians(0.53)   # đường kính đĩa Mặt Trời
    b_fine  = np.linspace(0.5, bl2.max() * 1.05, 700)
    x_j     = np.pi * b_fine * theta_s
    jinc_   = np.abs(2 * j1(x_j) / x_j)
    b_null  = 1.22 / theta_s   # first null ≈ 132 λ

    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.scatter(bl2, va2, s=6, color='royalblue', zorder=3,
               label='Simulated |V|  (forward model, Case 2)')
    ax.plot(b_fine, jinc_, color='darkorange', lw=1.8,
            label='Analytic jinc  (uniform disk, 0.53°)')
    ax.axvline(b_null, color='red', ls='--', lw=1.2,
               label=f'First null ≈ {b_null:.0f} λ  (≈ 65 m at 610 MHz)')
    ax.set_xlabel("Baseline length  (wavelengths λ)")
    ax.set_ylabel("Normalised visibility amplitude  |V(b)|")
    ax.set_title("Van Cittert–Zernike validation: visibility amplitude vs. baseline,  "
                 "20 March 2025")
    ax.legend(fontsize=8.5)
    ax.grid(True, alpha=0.3, ls='--')
    fig.tight_layout()
    _save("fig_visibility_jinc.png")
    del u2_, v2_, V2_
else:
    print("  [SKIP] fig_visibility_jinc — chạy Cell H trước (case2_data.npz chưa có)")

# ══════════════════════════════════════════════════════════════════════
# [6]  fig_dirty_image  —  Dirty image Case 1 riêng lẻ
#      Nguồn: img, l_arcmin, m_arcmin (Cell G)
# ══════════════════════════════════════════════════════════════════════
ext1     = [l_arcmin[0], l_arcmin[-1], m_arcmin[0], m_arcmin[-1]]
img_norm = img / img.max()

fig, ax = plt.subplots(figsize=(5.5, 5.2))
im = ax.imshow(img_norm, extent=ext1, origin='lower', cmap='hot',
               vmin=0, vmax=1, aspect='equal')
ax.plot(sun_r*np.cos(theta_c), sun_r*np.sin(theta_c),
        'c--', lw=1.5, label="Solar disk  (32′ diam.)")
ax.set_xlabel("East–West offset  (arcmin)")
ax.set_ylabel("North–South offset  (arcmin)")
ax.set_title("Dirty image — compact E–W array (Case 1),  20 March 2025")
ax.legend(fontsize=8, loc='upper right')
plt.colorbar(im, ax=ax, shrink=0.88, label='Normalised intensity')
fig.tight_layout()
_save("fig_dirty_image.png")

# ══════════════════════════════════════════════════════════════════════
# [7]  fig_dirty_beam  —  PSF Case 1 riêng lẻ
#      Nguồn: psf, fwhm_ew, fwhm_ns (Cell G)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(5.5, 5.2))
im = ax.imshow(psf, extent=ext1, origin='lower', cmap='RdBu_r',
               vmin=-0.5, vmax=1, aspect='equal')
ax.plot([-fwhm_ew/2, fwhm_ew/2], [0, 0], 'y-', lw=2.5,
        label=f"FWHM E–W = {fwhm_ew:.1f}′")
ax.plot([0, 0], [-fwhm_ns/2, fwhm_ns/2], 'c-', lw=2.5,
        label=f"FWHM N–S = {fwhm_ns:.1f}′")
ax.set_xlabel("East–West offset  (arcmin)")
ax.set_ylabel("North–South offset  (arcmin)")
ax.set_title("Synthesised dirty beam (PSF) — compact E–W array (Case 1)")
ax.legend(fontsize=8, loc='upper right')
plt.colorbar(im, ax=ax, shrink=0.88, label='Normalised PSF')
fig.tight_layout()
_save("fig_dirty_beam.png")

# ══════════════════════════════════════════════════════════════════════
# [8]  fig_sky_conv_psf  —  3 panel: sky model | sky*PSF | dirty image
#      Validate: (b) và (c) dùng CHUNG sky (đĩa đều 32′, đúng giả định
#      Engine B) trên chính u,v,w của Case 1 — kiểm tra code DFT
#      (make_dirty_image) tự nhất quán với định lý sky ⊛ PSF ≡ DFT(V_sky).
#      (c) ở đây là ảnh forward-model (Engine B: vis_uniform_disk), KHÔNG
#      phải img thật của Engine A (Cell C-E) — Engine A mô phỏng chuỗi
#      điện áp thật nên còn nhiễu Johnson-Nyquist + source_process ngẫu
#      nhiên (bức xạ Mặt Trời không kết hợp), không thể khớp pixel với
#      một mô hình xác định như (b). Muốn so Engine A thật, xem hình
#      fig_dirty_image.png / fig_dirty_beam.png ở trên (dùng img/psf thật).
#      Nguồn: u, v, w (Cell F), l_arcmin, m_arcmin, sun_r (Cell G)
# ══════════════════════════════════════════════════════════════════════
lg, mg   = np.meshgrid(l_arcmin, m_arcmin)
sky_mdl  = (np.sqrt(lg**2 + mg**2) <= sun_r).astype(float)   # đĩa đều 32′ (Engine B)

# PSF ở đây PHẢI cùng convention với img_disk (do_fringe_stop=False, kiểu
# Engine B thuần u-v) — không tái dùng biến "psf" của Cell G, vì psf đó
# dùng do_fringe_stop=True (đúng cho ảnh Engine A thật, sai convention nếu
# ghép với ảnh forward-model Engine B, làm (b)/(c) lệch pha giả tạo).
psf_b    = ix.make_dirty_image(
    u, v, w, np.ones_like(u), fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=False)[0]
psf_n    = psf_b / (psf_b.sum() + 1e-30)
sky_conv = fftconvolve(sky_mdl, psf_n, mode='same')
sky_conv = sky_conv / sky_conv.max()

# forward-model (Engine B) trên chính u,v,w của Case 1 — cùng sky với (a)/(b)
disk_diam_rad  = np.radians(2 * sun_r / 60)
V_disk         = ix.vis_uniform_disk(u, v, flux=1.0, diameter_rad=disk_diam_rad)
img_disk, _, _ = ix.make_dirty_image(
    u, v, w, V_disk, fov_deg=FOV_DEG, grid=GRID, do_fringe_stop=False)
img_disk_norm  = img_disk / img_disk.max()

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))
kw_im = dict(extent=ext1, origin='lower', cmap='hot', vmin=0, vmax=1, aspect='equal')
for ax_, data, title in zip(
    axes,
    [sky_mdl,   sky_conv,   img_disk_norm],
    ["(a)  True sky model\n(uniform disk, 32′ diam.)",
     "(b)  Sky ∗ PSF\n(independently computed)",
     "(c)  Dirty image from DFT\n(Engine B forward model, same u,v)"],
):
    im_ = ax_.imshow(data, **kw_im)
    ax_.set_title(title, fontsize=9.5)
    ax_.set_xlabel("E–W  (arcmin)", fontsize=8.5)
    ax_.set_ylabel("N–S  (arcmin)", fontsize=8.5)
    plt.colorbar(im_, ax=ax_, shrink=0.82, label='Norm. intensity')

fig.suptitle("Imaging validation: dirty image = sky ∗ PSF  "
             "(Case 1, compact E–W array)", fontsize=10.5)
fig.tight_layout()
_save("fig_sky_conv_psf.png")

# ══════════════════════════════════════════════════════════════════════
# [9]  fig_dirty_from_dft  —  Case 1.5 CROSS: 4 panel
#      Nguồn: img_c, psf_c, la, ma, fwhm_ew_c, fwhm_ns_c (Cell G2)
# ══════════════════════════════════════════════════════════════════════
ext_c     = [la[0], la[-1], ma[0], ma[-1]]
ic_c      = np.argmin(np.abs(la))
ir_c      = np.argmin(np.abs(ma))
img_c_n   = img_c / img_c.max()

fig = plt.figure(figsize=(11, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.32)

# (a) dirty image
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(img_c_n, extent=ext_c, origin='lower', cmap='hot',
                 vmin=0, vmax=1, aspect='equal')
ax1.plot(sun_r*np.cos(theta_c), sun_r*np.sin(theta_c), 'c--', lw=1.3)
ax1.set_title("(a)  Dirty image — cross layout (Case 1.5)")
ax1.set_xlabel("E–W  (arcmin)"); ax1.set_ylabel("N–S  (arcmin)")
plt.colorbar(im1, ax=ax1, shrink=0.87)

# (b) PSF
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(psf_c, extent=ext_c, origin='lower', cmap='RdBu_r',
                 vmin=-0.5, vmax=1, aspect='equal')
ax2.plot([-fwhm_ew_c/2, fwhm_ew_c/2], [0, 0], 'y-', lw=2.2,
         label=f"FWHM E–W = {fwhm_ew_c:.1f}′")
ax2.plot([0, 0], [-fwhm_ns_c/2, fwhm_ns_c/2], 'c-', lw=2.2,
         label=f"FWHM N–S = {fwhm_ns_c:.1f}′")
ax2.set_title(f"(b)  PSF — cross 4-ant")
ax2.set_xlabel("E–W  (arcmin)"); ax2.set_ylabel("N–S  (arcmin)")
ax2.legend(fontsize=7.5)
plt.colorbar(im2, ax=ax2, shrink=0.87)

# (c) E–W 1D cut
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(la, img_c_n[ir_c, :], color='darkorange', lw=1.8, label='Dirty image')
ax3.plot(la, psf_c[ir_c, :],   color='gray',       lw=1.2, ls='--', label='PSF')
ax3.axhline(0.5, color='gold', ls=':', lw=1, label=f"FWHM = {fwhm_ew_c:.1f}′")
ax3.set_xlabel("E–W offset  (arcmin)")
ax3.set_ylabel("Normalised intensity")
ax3.set_title("(c)  E–W profile through centre")
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3, ls='--')

# (d) N–S 1D cut
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(ma, img_c_n[:, ic_c], color='royalblue', lw=1.8, label='Dirty image')
ax4.plot(ma, psf_c[:, ic_c],   color='gray',      lw=1.2, ls='--', label='PSF')
ax4.axhline(0.5, color='cyan', ls=':', lw=1, label=f"FWHM = {fwhm_ns_c:.1f}′")
ax4.set_xlabel("N–S offset  (arcmin)")
ax4.set_ylabel("Normalised intensity")
ax4.set_title("(d)  N–S profile through centre")
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3, ls='--')

fig.suptitle(
    f"Case 1.5 — Cross 4-antenna layout (2D coverage)\n"
    f"N–S FWHM: {fwhm_ns:.1f}′ (Case 1, E–W only)  →  {fwhm_ns_c:.1f}′ "
    f"(Case 1.5, cross)  [{fwhm_ns/fwhm_ns_c:.1f}× improvement]",
    fontsize=10
)
_save("fig_dirty_from_dft.png")

# ══════════════════════════════════════════════════════════════════════
# [9.75] fig_case175_dirty_from_dft  —  Case 1.75 TAM GIÁC + TÂM (DLITE): 4 panel
#        Nguồn: img_t, psf_t, la_t, ma_t, fwhm_ew_t, fwhm_ns_t (Cell G3)
# ══════════════════════════════════════════════════════════════════════
ext_t     = [la_t[0], la_t[-1], ma_t[0], ma_t[-1]]
ic_t      = np.argmin(np.abs(la_t))
ir_t      = np.argmin(np.abs(ma_t))
img_t_n   = img_t / img_t.max()

fig = plt.figure(figsize=(11, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.32)

# (a) dirty image
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(img_t_n, extent=ext_t, origin='lower', cmap='hot',
                 vmin=0, vmax=1, aspect='equal')
ax1.plot(sun_r*np.cos(theta_c), sun_r*np.sin(theta_c), 'c--', lw=1.3)
ax1.set_title("(a)  Dirty image — triangle+centre layout (Case 1.75)")
ax1.set_xlabel("E–W  (arcmin)"); ax1.set_ylabel("N–S  (arcmin)")
plt.colorbar(im1, ax=ax1, shrink=0.87)

# (b) PSF
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(psf_t, extent=ext_t, origin='lower', cmap='RdBu_r',
                 vmin=-0.5, vmax=1, aspect='equal')
ax2.plot([-fwhm_ew_t/2, fwhm_ew_t/2], [0, 0], 'y-', lw=2.2,
         label=f"FWHM E–W = {fwhm_ew_t:.1f}′")
ax2.plot([0, 0], [-fwhm_ns_t/2, fwhm_ns_t/2], 'c-', lw=2.2,
         label=f"FWHM N–S = {fwhm_ns_t:.1f}′")
ax2.set_title(f"(b)  PSF — DLITE-style triangle 4-ant")
ax2.set_xlabel("E–W  (arcmin)"); ax2.set_ylabel("N–S  (arcmin)")
ax2.legend(fontsize=7.5)
plt.colorbar(im2, ax=ax2, shrink=0.87)

# (c) E–W 1D cut
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(la_t, img_t_n[ir_t, :], color='darkorange', lw=1.8, label='Dirty image')
ax3.plot(la_t, psf_t[ir_t, :],   color='gray',        lw=1.2, ls='--', label='PSF')
ax3.axhline(0.5, color='gold', ls=':', lw=1, label=f"FWHM = {fwhm_ew_t:.1f}′")
ax3.set_xlabel("E–W offset  (arcmin)")
ax3.set_ylabel("Normalised intensity")
ax3.set_title("(c)  E–W profile through centre")
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3, ls='--')

# (d) N–S 1D cut
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(ma_t, img_t_n[:, ic_t], color='royalblue', lw=1.8, label='Dirty image')
ax4.plot(ma_t, psf_t[:, ic_t],   color='gray',       lw=1.2, ls='--', label='PSF')
ax4.axhline(0.5, color='cyan', ls=':', lw=1, label=f"FWHM = {fwhm_ns_t:.1f}′")
ax4.set_xlabel("N–S offset  (arcmin)")
ax4.set_ylabel("Normalised intensity")
ax4.set_title("(d)  N–S profile through centre")
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3, ls='--')

fig.suptitle(
    f"Case 1.75 — DLITE-style triangle+centre layout (Helmboldt et al. 2021)\n"
    f"N–S FWHM: {fwhm_ns:.1f}′ (Case 1, E–W only)  →  {fwhm_ns_t:.1f}′ "
    f"(Case 1.75, triangle)  [{fwhm_ns/fwhm_ns_t:.1f}× improvement,  "
    f"{n_dir_tri}/6 baseline directions unique vs {n_dir_cross}/6 for cross]",
    fontsize=10
)
_save("fig_case175_dirty_from_dft.png")

# ══════════════════════════════════════════════════════════════════════
# [9.76] fig_case175_array_uv  —  Case 1.75: bố trí anten + UV coverage
#        Nguồn: ARRAY_TRI, u_t, v_t, NB_t, bl_labels_t (Cell G3)
# ══════════════════════════════════════════════════════════════════════
pos_E_t = ARRAY_TRI.positions_enu[:, 0]
pos_N_t = ARRAY_TRI.positions_enu[:, 1]
cmap_t  = plt.cm.tab10(np.linspace(0, 0.9, NB_t))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5))

for b, (i, j) in enumerate(ARRAY_TRI.pairs):
    axL.plot([pos_E_t[i], pos_E_t[j]], [pos_N_t[i], pos_N_t[j]],
             color=cmap_t[b], lw=1.8, label=bl_labels_t[b])
axL.scatter(pos_E_t, pos_N_t, s=140, color='white', edgecolor='navy', zorder=3)
for k in range(ARRAY_TRI.n_ant):
    axL.annotate(f"Ant {k}", (pos_E_t[k], pos_N_t[k]),
                 textcoords="offset points", xytext=(0, 8),
                 ha='center', fontsize=8, color='navy')
axL.set_xlabel("East  (m)"); axL.set_ylabel("North  (m)")
axL.set_title(f"(a)  Bố trí anten — tam giác đều + tâm (cạnh = {s:.0f} m)")
axL.set_aspect('equal'); axL.grid(True, alpha=0.3, ls='--')
axL.legend(fontsize=7, loc='upper right')

for b in range(NB_t):
    kw = dict(s=1.5, color=cmap_t[b], label=bl_labels_t[b])
    axR.scatter( u_t[:, b],  v_t[:, b], **kw)
    axR.scatter(-u_t[:, b], -v_t[:, b], s=1.5, color=cmap_t[b])
axR.set_xlabel("u  (wavelengths λ)"); axR.set_ylabel("v  (wavelengths λ)")
axR.set_title("(b)  UV coverage — triangle+centre (Case 1.75)")
axR.set_aspect('equal'); axR.grid(True, alpha=0.3, ls='--')
axR.legend(fontsize=7, markerscale=6, loc='upper right')

fig.suptitle("Case 1.75 — DLITE-style array geometry & UV coverage,  20 March 2025", fontsize=10.5)
fig.tight_layout()
_save("fig_case175_array_uv.png")

# ══════════════════════════════════════════════════════════════════════
# [10] fig_case2_image  —  Dirty image Case 2 (extended graded-cross)
#      Nguồn: case2_data.npz (Cell H) — img2 đã bị xoá trong Cell H
# ══════════════════════════════════════════════════════════════════════
if os.path.exists(npz_path2):
    d2_   = np.load(npz_path2)
    img2_ = d2_['img2']
    l2_a  = d2_['l2_deg'] * 60
    m2_a  = d2_['m2_deg'] * 60
    ext2  = [l2_a[0], l2_a[-1], m2_a[0], m2_a[-1]]

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    im = ax.imshow(img2_ / img2_.max(), extent=ext2, origin='lower', cmap='hot',
                   vmin=0, vmax=1, aspect='equal')
    ax.plot(sun_r*np.cos(theta_c), sun_r*np.sin(theta_c),
            'c--', lw=1.5, label="Solar disk  (32′ diam.)")
    ax.set_xlabel("East–West offset  (arcmin)")
    ax.set_ylabel("North–South offset  (arcmin)")
    ax.set_title("Dirty image — extended graded-cross (Case 2)\n"
                 "b_max = 130 m,  forward model,  610 MHz")
    ax.legend(fontsize=8, loc='upper right')
    plt.colorbar(im, ax=ax, shrink=0.88, label='Normalised intensity')
    fig.tight_layout()
    _save("fig_case2_image.png")
    del img2_
else:
    print("  [SKIP] fig_case2_image — chạy Cell H trước")

# ══════════════════════════════════════════════════════════════════════
# [11] fig_layout_sweep  —  copy _montage.png từ Cell I
#      Cell I phải chạy SAU cell này
# ══════════════════════════════════════════════════════════════════════
montage_src = os.path.join(IMG_DIR, "sweep_layouts", "_montage.png")
montage_dst = os.path.join(THESIS_FIG_DIR, "fig_layout_sweep.png")
if os.path.exists(montage_src):
    shutil.copy2(montage_src, montage_dst)
    print(f"  [OK] fig_layout_sweep.png  (copied from Cell I montage)")
else:
    print(f"  [SKIP] fig_layout_sweep — chạy Cell I trước, rồi chạy lại dòng copy này")

# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 55)
print(f"Hoàn tất! Ảnh đã lưu tại:")
print(f"  {THESIS_FIG_DIR}")
print()
print("Bước tiếp theo:")
print("  1. Kiểm tra các hình trong thư mục trên")
print("  2. Copy tất cả .png vào  d:\\datn\\figures\\")
print("  3. Biên dịch lại báo cáo (Ctrl+Alt+B)")
print("=" * 55)


Cell X: Bắt đầu xuất ảnh riêng lẻ cho báo cáo
  [OK] fig_sws_flux_day.png
  [OK] fig_voltage_env.png


C:\Users\admin\AppData\Local\Temp\ipykernel_4776\217858242.py:119: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  [OK] fig_fringe_pattern.png
  [OK] fig_uv_coverage.png
  [OK] fig_visibility_jinc.png
  [OK] fig_dirty_image.png
  [OK] fig_dirty_beam.png
  [OK] fig_sky_conv_psf.png
  [OK] fig_dirty_from_dft.png
  [OK] fig_case175_dirty_from_dft.png
  [OK] fig_case175_array_uv.png
  [OK] fig_case2_image.png
  [OK] fig_layout_sweep.png  (copied from Cell I montage)

Hoàn tất! Ảnh đã lưu tại:
  E:\1A DATN REAL\Cell_I_Data\thesis_figures

Bước tiếp theo:
  1. Kiểm tra các hình trong thư mục trên
  2. Copy tất cả .png vào  d:\datn\figures\
  3. Biên dịch lại báo cáo (Ctrl+Alt+B)


In [17]:
# %% CELL I: SWEEP — XUẤT ẢNH TĨNH XUỐNG Ổ E (không vẽ tương tác)
import interfsim as ix, numpy as np, os, gc
from math import radians
import matplotlib
matplotlib.use("Agg")                 # PHẢI đặt trước pyplot: không cửa sổ, không trình duyệt
import matplotlib.pyplot as plt

save_dir_I = os.path.join(IMG_DIR, "sweep_layouts")   # từ Cell 0
os.makedirs(save_dir_I, exist_ok=True)

sun_model = {'type': 'disk', 'diameter_rad': radians(0.53)}
try:
    _ = obs_img                       # dùng lại từ Cell H nếu đã chạy
except NameError:
    obs_img = ix.Observation(obs.lat_deg, obs.dec_deg, obs.fc_hz,
                             H_deg=np.linspace(obs.H_deg[0], obs.H_deg[-1], 600))
N_IMG  = obs_img.H_deg.size
bz_img = BEAM.response_voltage_offset(ix.zenith_angle_deg(obs_img)) ** 2

configs = [
    ix.preset_compact(),
    ix.linear(8, 15, az_deg=0),
    ix.tee(6, 5, 18),
    ix.preset_extended(),
    ix.random_2d(16, 120, seed=3),
]

FOV, GRID, TSTEP = 3.0, 192, 2        # TSTEP=2: nhẹ máy yếu; đặt 1 nếu muốn chất lượng tối đa
small = {}
summary = []

for cfg in configs:
    u, v, w, V = ix.forward_visibilities(cfg, obs_img, sun_model, time_weight=bz_img)
    img, l, m = ix.make_dirty_image(u, v, w, V, fov_deg=FOV, grid=GRID,
                                    do_fringe_stop=False, block=4096, time_step=TSTEP)
    psf, _, _ = ix.make_dirty_image(u, v, w, np.ones_like(V), fov_deg=FOV, grid=GRID,
                                    do_fringe_stop=False, block=4096, time_step=TSTEP)
    uf, vf = u.ravel(), v.ravel()
    blmax  = float(cfg.baseline_lengths.max())
    dr     = float(img.max() / img.std())     # dynamic range (peak/rms) để phân tích sau

    # --- lưu SỐ LIỆU ---
    np.savez(os.path.join(save_dir_I, f"{cfg.name}.npz"),
             img=img, psf=psf, u=uf, v=vf, l_deg=l, m_deg=m, bl_max=blmax)

    # --- lưu ẢNH từng layout (3 panel) ---
    ext = [l[0]*60, l[-1]*60, m[0]*60, m[-1]*60]
    th  = np.linspace(0, 2*np.pi, 200)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
    ax[0].imshow(img, extent=ext, origin='lower', cmap='hot', vmin=0, vmax=1)
    ax[0].plot(16*np.cos(th), 16*np.sin(th), 'c--', lw=1); ax[0].set_title("Dirty image")
    ax[1].imshow(psf, extent=ext, origin='lower', cmap='seismic', vmin=-1, vmax=1)
    ax[1].set_title("PSF (búp tổng hợp)")
    ax[2].scatter(np.r_[uf,-uf], np.r_[vf,-vf], s=1, c='royalblue', alpha=.4)
    ax[2].set_aspect('equal'); ax[2].set_title("u-v coverage")
    for a in ax[:2]: a.set_xlabel("Đông-Tây (')"); a.set_ylabel("Bắc-Nam (')")
    fig.suptitle(f"{cfg.name}  |  bl_max = {blmax:.0f} m  |  peak/rms = {dr:.1f}")
    fig.tight_layout(); fig.savefig(os.path.join(save_dir_I, f"{cfg.name}.png"), dpi=120)
    plt.close(fig)

    small[cfg.name] = dict(img=img, psf=psf, uf=uf, vf=vf, ext=ext, blmax=blmax)
    summary.append((cfg.name, blmax, dr))
    print(f"  [SAVE] {cfg.name}: bl_max={blmax:.0f} m, peak/rms={dr:.1f}")
    del u, v, w, V; gc.collect()

# --- MONTAGE tổng hợp 3 x 5 ---
n = len(configs)
fig, axs = plt.subplots(3, n, figsize=(3.0*n, 8.4))
for j, cfg in enumerate(configs):
    r = small[cfg.name]
    axs[0, j].imshow(r['img'], extent=r['ext'], origin='lower', cmap='hot', vmin=0, vmax=1)
    axs[0, j].set_title(f"{cfg.name}\nbl_max={r['blmax']:.0f} m", fontsize=8)
    axs[1, j].imshow(r['psf'], extent=r['ext'], origin='lower', cmap='seismic', vmin=-1, vmax=1)
    axs[2, j].scatter(np.r_[r['uf'],-r['uf']], np.r_[r['vf'],-r['vf']], s=.6, c='royalblue', alpha=.4)
    axs[2, j].set_aspect('equal')
    for i in range(3): axs[i, j].set_xticks([]); axs[i, j].set_yticks([])
for i, lab in enumerate(["Dirty", "PSF", "u-v"]): axs[i, 0].set_ylabel(lab, fontsize=11)
fig.suptitle(f"SWEEP — Dirty / PSF / u-v  (N_IMG={N_IMG}, time_step={TSTEP}, 610 MHz)")
fig.tight_layout(); fig.savefig(os.path.join(save_dir_I, "_montage.png"), dpi=130)
plt.close(fig)

# --- bảng tóm tắt để phân tích sau ---
with open(os.path.join(save_dir_I, "_summary.txt"), "w", encoding="utf-8") as f:
    f.write("layout\tbl_max_m\tpeak_over_rms\n")
    for nm, bl, dr in summary: f.write(f"{nm}\t{bl:.0f}\t{dr:.1f}\n")

del small; gc.collect()
print(f"\nCell I Xong! Ảnh + .npz + _montage.png + _summary.txt đã lưu tại:\n  {save_dir_I}")

  [SAVE] compact_outdoor_EW: bl_max=24 m, peak/rms=2.5
  [SAVE] linear_n8_d15_az0: bl_max=105 m, peak/rms=3.3
  [SAVE] tee_6x5_d18: bl_max=127 m, peak/rms=4.7
  [SAVE] extended_graded_cross_2D: bl_max=130 m, peak/rms=5.8
  [SAVE] random_n16_ext120_s3: bl_max=136 m, peak/rms=8.1

Cell I Xong! Ảnh + .npz + _montage.png + _summary.txt đã lưu tại:
  E:\1A DATN REAL\Cell_I_Data\sweep_layouts


---
## Đối chiếu với hệ thống vật lý
So sánh *solar excess above noise floor* giữa mô phỏng (Friis + 4NEC2 beam) và kết quả đo thực tế tại Hoa Lạc ngày 20-3-2025.  
Đại lượng so sánh: **excess = 10 log₁₀(1 + P_sig/P_noise)** — độc lập với gain chain phần cứng.

In [ ]:
# ============================================================
# ĐỐI CHIẼu VỚI HỆ THỐNG VẬT LÝ  (Hoa Lạc, 20-3-2025)
# Đại lượng so sánh: solar excess above noise floor (dB)
# Không phụ thuộc vào gain chain hardware
# ============================================================

# ── 1. Số đọc từ đồ thị đo thực ───────────────────────────────
# (đọc từ ảnh: 'Interferometry 610MHz at Hoa Lac - Ha Noi - 2025_03_20')
P_hw_noise_dB = -32.062   # noise floor trước transit (baseline)
P_hw_peak_dB  = -32.036   # đỉnh envelope transit ~12:00
excess_hw_dB  = P_hw_peak_dB - P_hw_noise_dB

# ── 2. Tính từ mô phỏng ──────────────────────────────────────
# beam_zenith = biên độ điện áp → ^2 = power response
P_sig_bw = P_signal * beam_zenith**2   # công suất có trọng số chùm tia

# Cửa sổ transit nhìn thấy trong đo thực: 10:00–14:00 ≈ |H| ≤ 30°
mask_hw         = np.abs(H_deg) <= 30.0
P_sig_peak_bw   = float(np.max(P_sig_bw[mask_hw]))
P_sig_avg_bw    = float(np.mean(P_sig_bw[mask_hw]))
sfu_avg_transit = float(np.mean(sfu_data[mask_hw]))

excess_sim_peak_dB = 10 * np.log10(1 + P_sig_peak_bw / P_noise)
excess_sim_avg_dB  = 10 * np.log10(1 + P_sig_avg_bw  / P_noise)

# ── 3. Bảng so sánh ──────────────────────────────────────────
sep = '=' * 62
dash = '-' * 62
print(sep)
print('  ĐỐI CHIẼu HỆ THỐNG VẬT LÝ vs MÔ PHỊNG  (20-3-2025)')
print(sep)
print(f'\n  Tham số hệ thống:')
print(f'    T_sys    = {T_sys:.0f} K')
print(f'    BW       = {BW_Hz/1e6:.0f} MHz')
print(f'    Aeff     = {Aeff*1e4:.2f} cm²')
print(f'    P_noise  = {P_noise:.4e} W  ({10*np.log10(P_noise*1e3):.2f} dBm)')
print(f'    SFU tb (|H|≤30°) = {sfu_avg_transit:.1f} SFU')

print(f'\n  Mô phỏng (Friis + 4NEC2 beam, cửa sổ |H| ≤ 30°):')
print(f'    P_sig đỉnh (beam-weighted) = {P_sig_peak_bw:.4e} W')
print(f'    P_sig tb   (beam-weighted) = {P_sig_avg_bw:.4e} W')
print(f'    Excess đỉnh transit        = {excess_sim_peak_dB:.4f} dB')
print(f'    Excess trung bình          = {excess_sim_avg_dB:.4f} dB')

print(f'\n  Đo thực tế (Hoa Lạc, 20-3-2025):')
print(f'    Noise floor (baseline)     = {P_hw_noise_dB:.3f} dB')
print(f'    Đỉnh envelope transit      = {P_hw_peak_dB:.3f} dB')
print(f'    Excess đo được             = {excess_hw_dB:.4f} dB')

print(f'\n  Tỉ số sim vs meas (dB):')
diff_peak = excess_sim_peak_dB - excess_hw_dB
diff_avg  = excess_sim_avg_dB  - excess_hw_dB
print(f'    excess_sim(đỉnh) − excess_hw = {diff_peak:+.3f} dB')
print(f'    excess_sim(tb)   − excess_hw = {diff_avg:+.3f} dB')

print(f'\n{dash}')
print(f'  → Giá trị điền vào Table 5.7 (luận văn):')
print(f'    P_sim  = {excess_sim_avg_dB:.4f} dB above noise  (transit avg, |H|≤30°)')
print(f'    P_meas = {excess_hw_dB:.4f} dB above noise  (đỉnh envelope, đồ thị)')
print(f'    Tỉ số  = {diff_avg:+.3f} dB')
print(f'{dash}')


  ĐỐI CHIẼu HỆ THỐNG VẬT LÝ vs MÔ PHỊNG  (20-3-2025)

  Tham số hệ thống:
    T_sys    = 150 K
    BW       = 1 MHz
    Aeff     = 1705.19 cm²
    P_noise  = 2.0710e-15 W  (-116.84 dBm)
    SFU tb (|H|≤30°) = 69.3 SFU

  Mô phỏng (Friis + 4NEC2 beam, cửa sổ |H| ≤ 30°):
    P_sig đỉnh (beam-weighted) = 7.3347e-16 W
    P_sig tb   (beam-weighted) = 5.4974e-16 W
    Excess đỉnh transit        = 1.3167 dB
    Excess trung bình          = 1.0225 dB

  Đo thực tế (Hoa Lạc, 20-3-2025):
    Noise floor (baseline)     = -32.062 dB
    Đỉnh envelope transit      = -32.036 dB
    Excess đo được             = 0.0260 dB

  Tỉ số sim vs meas (dB):
    excess_sim(đỉnh) − excess_hw = +1.291 dB
    excess_sim(tb)   − excess_hw = +0.996 dB

--------------------------------------------------------------
  → Giá trị điền vào Table 5.7 (luận văn):
    P_sim  = 1.0225 dB above noise  (transit avg, |H|≤30°)
    P_meas = 0.0260 dB above noise  (đỉnh envelope, đồ thị)
    Tỉ số  = +0.996 dB
-------------------

: 